# EXECUÇÃO DOS MODELOS COMPARATIVOS - TODOS OS EXPERIMENTOS

Este notebook executa todos os modelos comparativos (HAT, ARF, SRP, ACDWM, ERulesD2S, ROSE) em **todos os 3 experimentos**:

| Experimento | chunk_size | PENALTY_WEIGHT | Diretórios |
|-------------|------------|----------------|------------|
| **EXP-A** | 1000 | 0.0 | experiments_6chunks_phase2_gbml, experiments_6chunks_phase3_real |
| **EXP-B** | 2000 | 0.0 | experiments_chunk2000_phase1, experiments_chunk2000_phase2 |
| **EXP-C** | 2000 | 0.1 | experiments_balanced_phase1, experiments_balanced_phase2 |

**Modelos:**
- ROSE_Original (prequential, como no paper)
- ROSE_ChunkEval (avaliação por chunk, comparável com GBML)
- HAT (Hoeffding Adaptive Tree)
- ARF (Adaptive Random Forest)
- SRP (Streaming Random Patches)
- ACDWM (apenas binário)
- ERulesD2S (cache existente)

**Importante:** Os resultados são salvos nas mesmas pastas do GBML para facilitar a comparação.

---
## PARTE 1: SETUP DO AMBIENTE
---

In [ ]:
# CÉLULA 1.1: Montar Google Drive
from google.colab import drive
import os
import sys
from pathlib import Path

# Montar Drive
drive.mount('/content/drive')

# Diretório de trabalho no Drive
DRIVE_PATH = '/content/drive/MyDrive/DSL-AG-hybrid'
WORK_DIR = '/content/drive/Othercomputers/Laptop-CIn/Downloads/DSL-AG-hybrid'

# Copiar para /content (mais rápido)
# if not os.path.exists(WORK_DIR):
#     print(f"Copiando {DRIVE_PATH} para {WORK_DIR}...")
#     !cp -r "{DRIVE_PATH}" "{WORK_DIR}"
#     print("Cópia concluída!")
# else:
#     print(f"Diretório {WORK_DIR} já existe.")

os.chdir(WORK_DIR)
print(f"Diretório de trabalho: {os.getcwd()}")

Mounted at /content/drive
Diretório de trabalho: /content/drive/Othercomputers/Laptop-CIn/Downloads/DSL-AG-hybrid


In [ ]:
# CÉLULA 1.2: Instalar Java 11 e Maven (necessário para ROSE e ERulesD2S)
#
# Java 11 é requerido pelo ERulesD2S (JCLEC4)
# Maven é necessário para compilar dependências Java

print("Instalando Java 11 e Maven...")

!apt-get update -qq
!apt-get install -y -qq openjdk-11-jdk maven > /dev/null 2>&1

# Configurar JAVA_HOME (importante para ERulesD2S)
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

# Verificar instalações
print("\n--- Java ---")
!java -version

print("\n--- Maven ---")
!mvn -version 2>/dev/null | head -3

print("\nJava 11 e Maven instalados com sucesso!")
print(f"JAVA_HOME: {os.environ.get('JAVA_HOME', 'NÃO DEFINIDO')}")

Instalando Java 11 e Maven...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

--- Java ---
openjdk version "11.0.29" 2025-10-21
OpenJDK Runtime Environment (build 11.0.29+7-post-Ubuntu-1ubuntu122.04)
OpenJDK 64-Bit Server VM (build 11.0.29+7-post-Ubuntu-1ubuntu122.04, mixed mode, sharing)

--- Maven ---
Apache Maven 3.6.3
Maven home: /usr/share/maven
Java version: 11.0.29, vendor: Ubuntu, runtime: /usr/lib/jvm/java-11-openjdk-amd64

Java 11 e Maven instalados com sucesso!
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64


In [ ]:
# CÉLULA 1.3: Instalar Dependências Python
print("Instalando dependências Python...")
!pip install -q river scikit-learn pandas numpy scipy scikit-posthocs matplotlib seaborn
print("Dependências instaladas!")

Instalando dependências Python...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 115.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
Dependências instaladas!


In [ ]:
# CÉLULA 1.4: Baixar JARs do ROSE
import urllib.request
from pathlib import Path

rose_dir = Path('rose_jars')
rose_dir.mkdir(exist_ok=True)

# URLs dos JARs
ROSE_JARS = {
    'ROSE-1.0.jar': 'https://github.com/canoalberto/ROSE/releases/download/v1.0/ROSE-1.0.jar',
    'MOA-dependencies.jar': 'https://github.com/canoalberto/ROSE/releases/download/v1.0/MOA-dependencies.jar',
    'sizeofag-1.0.4.jar': 'https://repo1.maven.org/maven2/com/github/fracpete/sizeofag/1.0.4/sizeofag-1.0.4.jar'
}

for jar_name, url in ROSE_JARS.items():
    jar_path = rose_dir / jar_name
    if not jar_path.exists():
        print(f"Baixando {jar_name}...")
        urllib.request.urlretrieve(url, jar_path)
    print(f"  {jar_name}: OK")

print("\nROSE configurado!")

  ROSE-1.0.jar: OK
  MOA-dependencies.jar: OK
  sizeofag-1.0.4.jar: OK

ROSE configurado!


In [ ]:
# CÉLULA 1.5: Clonar ACDWM Repository
ACDWM_DIR = "/content/ACDWM"

if not os.path.exists(ACDWM_DIR):
    print("Clonando repositório ACDWM...")
    !git clone https://github.com/jasonyanglu/ACDWM.git {ACDWM_DIR}
else:
    print("ACDWM já clonado.")

# Verificar arquivos
dwmil_file = Path(ACDWM_DIR) / "dwmil.py"
print(f"dwmil.py: {'OK' if dwmil_file.exists() else 'FALTANDO'}")

Clonando repositório ACDWM...
Cloning into '/content/ACDWM'...
remote: Enumerating objects: 117, done.
remote: Total 117 (delta 0), reused 0 (delta 0), pack-reused 117 (from 1)
Receiving objects: 100% (117/117), 14.80 MiB | 36.78 MiB/s, done.
Resolving deltas: 100% (26/26), done.
dwmil.py: OK


In [ ]:
# CÉLULA 1.6: Setup ERulesD2S
#
# ERulesD2S requer:
# - Java 11 (instalado na célula anterior)
# - Maven (instalado na célula anterior)
# - erulesd2s.jar
# - JCLEC4-base-1.0-jar-with-dependencies.jar (na pasta lib/)

print("="*60)
print("CONFIGURANDO ERulesD2S")
print("="*60)

# Verificar JAVA_HOME
java_home = os.environ.get('JAVA_HOME', '')
if java_home:
    print(f"JAVA_HOME: {java_home}")
else:
    print("AVISO: JAVA_HOME não definido - definindo agora...")
    os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
    print(f"JAVA_HOME: {os.environ['JAVA_HOME']}")

# Verificar se JARs existem
erulesd2s_jar = Path(WORK_DIR) / "erulesd2s.jar"
jclec_jar = Path(WORK_DIR) / "lib" / "JCLEC4-base-1.0-jar-with-dependencies.jar"

print(f"\nVerificando JARs...")
print(f"  erulesd2s.jar: {'ENCONTRADO' if erulesd2s_jar.exists() else 'NÃO ENCONTRADO'}")
print(f"  JCLEC4 JAR: {'ENCONTRADO' if jclec_jar.exists() else 'NÃO ENCONTRADO'}")

# Se JAR não existe, executar setup
if not erulesd2s_jar.exists():
    print("\nExecutando setup_erulesd2s.py...")
    setup_script = Path(WORK_DIR) / "setup_erulesd2s.py"

    if setup_script.exists():
        # Executar setup (pode demorar alguns minutos)
        !cd {WORK_DIR} && python setup_erulesd2s.py

        # Verificar novamente
        if erulesd2s_jar.exists():
            print("\nERulesD2S configurado com sucesso!")
            print(f"  JAR: {erulesd2s_jar}")

            # Verificar JCLEC4
            if jclec_jar.exists():
                print(f"  JCLEC4: {jclec_jar}")
            else:
                print("  AVISO: JCLEC4 JAR não encontrado - ERulesD2S pode falhar!")
        else:
            print("\nERRO: erulesd2s.jar não foi criado!")
            print("ERulesD2S será ignorado nos experimentos.")
    else:
        print(f"\nERRO: {setup_script} não encontrado!")
        print("ERulesD2S será ignorado nos experimentos.")
else:
    print("\nERulesD2S já está configurado!")

    # Verificar tamanho do JAR (deve ser grande, ~58MB)
    jar_size = erulesd2s_jar.stat().st_size / (1024 * 1024)
    print(f"  Tamanho: {jar_size:.1f} MB")

    # Verificar JCLEC4 mesmo assim
    if not jclec_jar.exists():
        print("  AVISO: JCLEC4 JAR não encontrado - ERulesD2S pode falhar!")
    else:
        jclec_size = jclec_jar.stat().st_size / (1024 * 1024)
        print(f"  JCLEC4: {jclec_size:.1f} MB")

# Listar pasta lib se existir
lib_dir = Path(WORK_DIR) / "lib"
if lib_dir.exists():
    print(f"\nConteúdo da pasta lib/:")
    !ls -lh {lib_dir}/*.jar 2>/dev/null | head -5

# Teste rápido: verificar se Java consegue carregar o JAR
if erulesd2s_jar.exists():
    print("\nTeste rápido do JAR...")
    test_result = !java -cp {erulesd2s_jar} moa.DoTask 2>&1 | head -3
    if test_result:
        print("  JAR carregável: OK")
    else:
        print("  AVISO: Não foi possível verificar o JAR")

print("\n" + "="*60)

CONFIGURANDO ERulesD2S
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64

Verificando JARs...
  erulesd2s.jar: ENCONTRADO
  JCLEC4 JAR: ENCONTRADO

ERulesD2S já está configurado!
  Tamanho: 58.4 MB
  JCLEC4: 29.3 MB

Conteúdo da pasta lib/:
-rw------- 1 root root 30M Nov 14 00:55 /content/drive/Othercomputers/Laptop-CIn/Downloads/DSL-AG-hybrid/lib/JCLEC4-base-1.0-jar-with-dependencies.jar

Teste rápido do JAR...
  JAR carregável: OK



In [ ]:
# CÉLULA 1.6: Verificação do Ambiente
print("="*80)
print("VERIFICAÇÃO DO AMBIENTE")
print("="*80)

# 1. Java
print("\n1. Java")
!java -version 2>&1 | head -1

# 2. ROSE JARs
print("\n2. ROSE JARs")
for jar in ['ROSE-1.0.jar', 'MOA-dependencies.jar', 'sizeofag-1.0.4.jar']:
    exists = (Path('rose_jars') / jar).exists()
    print(f"   {jar}: {'OK' if exists else 'FALTANDO'}")

# 3. ACDWM
print("\n3. ACDWM")
print(f"   dwmil.py: {'OK' if Path(ACDWM_DIR + '/dwmil.py').exists() else 'FALTANDO'}")

# 4. River
print("\n4. River")
try:
    import river
    from river import tree, ensemble, forest
    print(f"   River version: {river.__version__}")
    print(f"   tree.HoeffdingAdaptiveTreeClassifier: OK")
    print(f"   forest.ARFClassifier: OK")
    print(f"   ensemble.SRPClassifier: OK")
except Exception as e:
    print(f"   River: ERRO - {e}")

# 5. ERulesD2S
print("\n5. ERulesD2S")
erulesd2s_jar = Path(WORK_DIR) / "erulesd2s.jar"
print(f"   erulesd2s.jar: {'OK' if erulesd2s_jar.exists() else 'FALTANDO (será ignorado)'}")

print("\n" + "="*80)

VERIFICAÇÃO DO AMBIENTE

1. Java
openjdk version "11.0.29" 2025-10-21

2. ROSE JARs
   ROSE-1.0.jar: OK
   MOA-dependencies.jar: OK
   sizeofag-1.0.4.jar: OK

3. ACDWM
   dwmil.py: OK

4. River
   River version: 0.23.0
   tree.HoeffdingAdaptiveTreeClassifier: OK
   forest.ARFClassifier: OK
   ensemble.SRPClassifier: OK

5. ERulesD2S
   erulesd2s.jar: OK



---
## PARTE 2: CONFIGURAÇÃO DOS EXPERIMENTOS
---

In [ ]:
# =============================================================================
# CELULA 2.1: Configuracao dos Experimentos (UNIFIED + ORIGINAIS)
# =============================================================================
# SUBSTITUA a CELULA 2.1 original por este codigo
# Adiciona experimentos unified (chunk_500, chunk_1000) com suporte a penalty
# Mantem experimentos originais (exp_a, exp_b, exp_c) para compatibilidade
# =============================================================================

import numpy as np
import pandas as pd
import json
import subprocess
import time
from datetime import datetime
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# PATHS IMPORTANTES
# =============================================================================
UNIFIED_CHUNKS_DIR = Path(WORK_DIR) / "unified_chunks"
EXPERIMENTS_UNIFIED_DIR = Path(WORK_DIR) / "experiments_unified"

# =============================================================================
# DATASETS MULTICLASSE (CDCMS N/A, ACDWM N/A)
# =============================================================================
MULTICLASS_DATASETS = {
    'LED_Abrupt_Simple': 10,
    'LED_Gradual_Simple': 10,
    'LED_Stationary': 10,
    'WAVEFORM_Abrupt_Simple': 3,
    'WAVEFORM_Gradual_Simple': 3,
    'WAVEFORM_Stationary': 3,
    'CovType': 7,
    'Shuttle': 7,
    'RBF_Stationary': 4,
}

# =============================================================================
# CONFIGURACAO DOS EXPERIMENTOS UNIFIED (NOVOS)
# =============================================================================

UNIFIED_EXPERIMENT_CONFIGS = {
    # =========================================================================
    # CHUNK 500 - SEM PENALTY (foco em performance)
    # =========================================================================
    'exp_unified_500': {
        'chunk_size': 500,
        'penalty_weight': 0.0,
        'data_source': 'unified',
        'data_dir': 'chunk_500',
        'results_dir': 'chunk_500',
        'egis_model_name': 'EGIS',
        'description': 'Unified chunks 500 (sem penalty)',
        'batches': {
            'batch_1': {
                'base_dir': 'experiments_unified/chunk_500/batch_1',
                'datasets': [
                    'SEA_Abrupt_Simple', 'SEA_Abrupt_Chain', 'SEA_Abrupt_Recurring',
                    'SEA_Gradual_Simple_Fast', 'SEA_Gradual_Simple_Slow', 'SEA_Gradual_Recurring',
                    'AGRAWAL_Abrupt_Simple_Mild', 'AGRAWAL_Abrupt_Simple_Severe', 'AGRAWAL_Abrupt_Chain_Long',
                    'HYPERPLANE_Abrupt_Simple',
                    'RANDOMTREE_Abrupt_Simple',
                    'RBF_Abrupt_Severe', 'RBF_Abrupt_Blip', 'RBF_Gradual_Moderate', 'RBF_Gradual_Severe',
                    'STAGGER_Abrupt_Chain', 'STAGGER_Abrupt_Recurring', 'STAGGER_Gradual_Chain'
                ]
            },
            'batch_2': {
                'base_dir': 'experiments_unified/chunk_500/batch_2',
                'datasets': [
                    'SEA_Abrupt_Chain_Noise',
                    'AGRAWAL_Abrupt_Simple_Severe_Noise',
                    'HYPERPLANE_Gradual_Simple', 'HYPERPLANE_Gradual_Noise',
                    'RANDOMTREE_Gradual_Simple', 'RANDOMTREE_Abrupt_Recurring', 'RANDOMTREE_Gradual_Noise',
                    'RBF_Abrupt_Blip_Noise', 'RBF_Gradual_Severe_Noise',
                    'STAGGER_Abrupt_Chain_Noise',
                    'LED_Abrupt_Simple', 'LED_Gradual_Simple',
                    'WAVEFORM_Abrupt_Simple', 'WAVEFORM_Gradual_Simple',
                    'SINE_Abrupt_Simple', 'SINE_Gradual_Recurring', 'SINE_Abrupt_Recurring_Noise'
                ]
            },
            'batch_3': {
                'base_dir': 'experiments_unified/chunk_500/batch_3',
                'datasets': [
                    'SEA_Stationary', 'AGRAWAL_Stationary', 'HYPERPLANE_Stationary',
                    'RANDOMTREE_Stationary', 'RBF_Stationary', 'STAGGER_Stationary',
                    'LED_Stationary', 'WAVEFORM_Stationary', 'SINE_Stationary',
                    'Electricity', 'Shuttle', 'CovType', 'PokerHand', 'IntelLabSensors',
                    'AssetNegotiation_F2', 'AssetNegotiation_F3', 'AssetNegotiation_F4'
                ]
            }
        }
    },

    # =========================================================================
    # CHUNK 500 - COM PENALTY (foco em interpretabilidade)
    # =========================================================================
    'exp_unified_500_penalty': {
        'chunk_size': 500,
        'penalty_weight': 0.1,
        'data_source': 'unified',
        'data_dir': 'chunk_500',  # Dados sao os mesmos
        'results_dir': 'chunk_500_penalty',  # Resultados EGIS diferentes
        'egis_model_name': 'EGIS_Penalty',
        'description': 'Unified chunks 500 (com penalty)',
        'reuse_comparative_from': 'exp_unified_500',  # Reutiliza modelos comparativos
        'batches': {
            'batch_1': {
                'base_dir': 'experiments_unified/chunk_500_penalty/batch_1',
                'datasets': [
                    'SEA_Abrupt_Simple', 'SEA_Abrupt_Chain', 'SEA_Abrupt_Recurring',
                    'SEA_Gradual_Simple_Fast', 'SEA_Gradual_Simple_Slow', 'SEA_Gradual_Recurring',
                    'AGRAWAL_Abrupt_Simple_Mild', 'AGRAWAL_Abrupt_Simple_Severe', 'AGRAWAL_Abrupt_Chain_Long',
                    'HYPERPLANE_Abrupt_Simple',
                    'RANDOMTREE_Abrupt_Simple',
                    'RBF_Abrupt_Severe', 'RBF_Abrupt_Blip', 'RBF_Gradual_Moderate', 'RBF_Gradual_Severe',
                    'STAGGER_Abrupt_Chain', 'STAGGER_Abrupt_Recurring', 'STAGGER_Gradual_Chain'
                ]
            },
            'batch_2': {
                'base_dir': 'experiments_unified/chunk_500_penalty/batch_2',
                'datasets': [
                    'SEA_Abrupt_Chain_Noise',
                    'AGRAWAL_Abrupt_Simple_Severe_Noise',
                    'HYPERPLANE_Gradual_Simple', 'HYPERPLANE_Gradual_Noise',
                    'RANDOMTREE_Gradual_Simple', 'RANDOMTREE_Abrupt_Recurring', 'RANDOMTREE_Gradual_Noise',
                    'RBF_Abrupt_Blip_Noise', 'RBF_Gradual_Severe_Noise',
                    'STAGGER_Abrupt_Chain_Noise',
                    'LED_Abrupt_Simple', 'LED_Gradual_Simple',
                    'WAVEFORM_Abrupt_Simple', 'WAVEFORM_Gradual_Simple',
                    'SINE_Abrupt_Simple', 'SINE_Gradual_Recurring', 'SINE_Abrupt_Recurring_Noise'
                ]
            },
            'batch_3': {
                'base_dir': 'experiments_unified/chunk_500_penalty/batch_3',
                'datasets': [
                    'SEA_Stationary', 'AGRAWAL_Stationary', 'HYPERPLANE_Stationary',
                    'RANDOMTREE_Stationary', 'RBF_Stationary', 'STAGGER_Stationary',
                    'LED_Stationary', 'WAVEFORM_Stationary', 'SINE_Stationary',
                    'Electricity', 'Shuttle', 'CovType', 'PokerHand', 'IntelLabSensors',
                    'AssetNegotiation_F2', 'AssetNegotiation_F3', 'AssetNegotiation_F4'
                ]
            }
        }
    },

    # =========================================================================
    # CHUNK 1000 - SEM PENALTY (foco em performance)
    # =========================================================================
    'exp_unified_1000': {
        'chunk_size': 1000,
        'penalty_weight': 0.0,
        'data_source': 'unified',
        'data_dir': 'chunk_1000',
        'results_dir': 'chunk_1000',
        'egis_model_name': 'EGIS',
        'description': 'Unified chunks 1000 (sem penalty)',
        'batches': {
            'batch_1': {
                'base_dir': 'experiments_unified/chunk_1000/batch_1',
                'datasets': [
                    'SEA_Abrupt_Simple', 'SEA_Abrupt_Chain', 'SEA_Abrupt_Recurring', 'SEA_Gradual_Simple_Fast',
                    'AGRAWAL_Abrupt_Simple_Mild', 'AGRAWAL_Abrupt_Simple_Severe', 'AGRAWAL_Abrupt_Chain_Long',
                    'HYPERPLANE_Abrupt_Simple',
                    'RANDOMTREE_Abrupt_Simple',
                    'RBF_Abrupt_Blip', 'RBF_Abrupt_Severe',
                    'STAGGER_Abrupt_Chain', 'STAGGER_Abrupt_Recurring'
                ]
            },
            'batch_2': {
                'base_dir': 'experiments_unified/chunk_1000/batch_2',
                'datasets': [
                    'SEA_Abrupt_Chain_Noise', 'SEA_Gradual_Recurring', 'SEA_Gradual_Simple_Slow',
                    'AGRAWAL_Abrupt_Simple_Severe_Noise',
                    'HYPERPLANE_Gradual_Simple',
                    'RANDOMTREE_Gradual_Simple',
                    'RBF_Abrupt_Blip_Noise', 'RBF_Gradual_Moderate', 'RBF_Gradual_Severe',
                    'STAGGER_Abrupt_Chain_Noise', 'STAGGER_Gradual_Chain',
                    'LED_Gradual_Simple',
                    'SINE_Abrupt_Recurring_Noise'
                ]
            },
            'batch_3': {
                'base_dir': 'experiments_unified/chunk_1000/batch_3',
                'datasets': [
                    'HYPERPLANE_Gradual_Noise',
                    'RANDOMTREE_Abrupt_Recurring', 'RANDOMTREE_Gradual_Noise',
                    'RBF_Gradual_Severe_Noise',
                    'LED_Abrupt_Simple',
                    'WAVEFORM_Abrupt_Simple', 'WAVEFORM_Gradual_Simple',
                    'SINE_Abrupt_Simple', 'SINE_Gradual_Recurring',
                    'Electricity', 'Shuttle', 'CovType', 'PokerHand'
                ]
            },
            'batch_4': {
                'base_dir': 'experiments_unified/chunk_1000/batch_4',
                'datasets': [
                    'SEA_Stationary', 'AGRAWAL_Stationary', 'HYPERPLANE_Stationary',
                    'RANDOMTREE_Stationary', 'RBF_Stationary', 'STAGGER_Stationary',
                    'LED_Stationary', 'WAVEFORM_Stationary', 'SINE_Stationary',
                    'IntelLabSensors',
                    'AssetNegotiation_F2', 'AssetNegotiation_F3', 'AssetNegotiation_F4'
                ]
            }
        }
    },

    # =========================================================================
    # CHUNK 1000 - COM PENALTY (foco em interpretabilidade)
    # =========================================================================
    'exp_unified_1000_penalty': {
        'chunk_size': 1000,
        'penalty_weight': 0.1,
        'data_source': 'unified',
        'data_dir': 'chunk_1000',  # Dados sao os mesmos
        'results_dir': 'chunk_1000_penalty',  # Resultados EGIS diferentes
        'egis_model_name': 'EGIS_Penalty',
        'description': 'Unified chunks 1000 (com penalty)',
        'reuse_comparative_from': 'exp_unified_1000',  # Reutiliza modelos comparativos
        'batches': {
            'batch_1': {
                'base_dir': 'experiments_unified/chunk_1000_penalty/batch_1',
                'datasets': [
                    'SEA_Abrupt_Simple', 'SEA_Abrupt_Chain', 'SEA_Abrupt_Recurring', 'SEA_Gradual_Simple_Fast',
                    'AGRAWAL_Abrupt_Simple_Mild', 'AGRAWAL_Abrupt_Simple_Severe', 'AGRAWAL_Abrupt_Chain_Long',
                    'HYPERPLANE_Abrupt_Simple',
                    'RANDOMTREE_Abrupt_Simple',
                    'RBF_Abrupt_Blip', 'RBF_Abrupt_Severe',
                    'STAGGER_Abrupt_Chain', 'STAGGER_Abrupt_Recurring'
                ]
            },
            'batch_2': {
                'base_dir': 'experiments_unified/chunk_1000_penalty/batch_2',
                'datasets': [
                    'SEA_Abrupt_Chain_Noise', 'SEA_Gradual_Recurring', 'SEA_Gradual_Simple_Slow',
                    'AGRAWAL_Abrupt_Simple_Severe_Noise',
                    'HYPERPLANE_Gradual_Simple',
                    'RANDOMTREE_Gradual_Simple',
                    'RBF_Abrupt_Blip_Noise', 'RBF_Gradual_Moderate', 'RBF_Gradual_Severe',
                    'STAGGER_Abrupt_Chain_Noise', 'STAGGER_Gradual_Chain',
                    'LED_Gradual_Simple',
                    'SINE_Abrupt_Recurring_Noise'
                ]
            },
            'batch_3': {
                'base_dir': 'experiments_unified/chunk_1000_penalty/batch_3',
                'datasets': [
                    'HYPERPLANE_Gradual_Noise',
                    'RANDOMTREE_Abrupt_Recurring', 'RANDOMTREE_Gradual_Noise',
                    'RBF_Gradual_Severe_Noise',
                    'LED_Abrupt_Simple',
                    'WAVEFORM_Abrupt_Simple', 'WAVEFORM_Gradual_Simple',
                    'SINE_Abrupt_Simple', 'SINE_Gradual_Recurring',
                    'Electricity', 'Shuttle', 'CovType', 'PokerHand'
                ]
            },
            'batch_4': {
                'base_dir': 'experiments_unified/chunk_1000_penalty/batch_4',
                'datasets': [
                    'SEA_Stationary', 'AGRAWAL_Stationary', 'HYPERPLANE_Stationary',
                    'RANDOMTREE_Stationary', 'RBF_Stationary', 'STAGGER_Stationary',
                    'LED_Stationary', 'WAVEFORM_Stationary', 'SINE_Stationary',
                    'IntelLabSensors',
                    'AssetNegotiation_F2', 'AssetNegotiation_F3', 'AssetNegotiation_F4'
                ]
            }
        }
    }
}

# =============================================================================
# COMBINAR CONFIGS (ORIGINAIS + UNIFIED)
# =============================================================================
# Manter experimentos originais para compatibilidade
EXPERIMENT_CONFIGS = {
    'exp_a_chunk1000': {
        'chunk_size': 1000,
        'penalty_weight': 0.0,
        'data_source': 'legacy',  # Fonte antiga
        'description': 'Baseline configuration (chunk_size=1000)',
        'batches': {
            'batch_1': {
                'base_dir': 'experiments_6chunks_phase2_gbml/batch_1',
                'datasets': [
                    'SEA_Abrupt_Simple', 'SEA_Abrupt_Chain', 'SEA_Abrupt_Recurring',
                    'AGRAWAL_Abrupt_Simple_Mild', 'AGRAWAL_Abrupt_Simple_Severe', 'AGRAWAL_Abrupt_Chain_Long',
                    'RBF_Abrupt_Severe', 'RBF_Abrupt_Blip',
                    'STAGGER_Abrupt_Chain', 'STAGGER_Abrupt_Recurring',
                    'HYPERPLANE_Abrupt_Simple', 'RANDOMTREE_Abrupt_Simple'
                ]
            }
            # ... outros batches mantidos igual ao original
        }
    }
    # ... exp_b_chunk2000 e exp_c_balanced mantidos igual ao original
}

# Adicionar experimentos unified
EXPERIMENT_CONFIGS.update(UNIFIED_EXPERIMENT_CONFIGS)

# =============================================================================
# SELECIONAR EXPERIMENTOS A EXECUTAR
# =============================================================================
# Descomente os experimentos que deseja executar:

EXPERIMENTS_TO_RUN = [
    # Novos experimentos unified
    'exp_unified_500',           # Chunk 500 sem penalty
    'exp_unified_500_penalty',   # Chunk 500 com penalty
    'exp_unified_1000',          # Chunk 1000 sem penalty
    'exp_unified_1000_penalty',  # Chunk 1000 com penalty

    # Experimentos legados (descomente se necessario)
    # 'exp_a_chunk1000',
    # 'exp_b_chunk2000',
    # 'exp_c_balanced',
]

# =============================================================================
# MODELOS A EXECUTAR
# =============================================================================
# EGIS e CDCMS serao carregados de cache (ja executados)
# Demais modelos serao executados em tempo real

MODELS_TO_RUN = [
    'ROSE_Original',    # ROSE como no paper (metrica global)
    'ROSE_ChunkEval',   # ROSE com avaliacao por chunk
    'HAT',              # Hoeffding Adaptive Tree (River)
    'ARF',              # Adaptive Random Forest (River)
    'SRP',              # Streaming Random Patches (River)
    'ACDWM',            # Adaptive Chunk-based DWM (so binario!)
    'ERulesD2S'       # Desabilitado por ser lento
]

# Timeout por modelo (segundos)
MODEL_TIMEOUT = {
    'ROSE_Original': 600,
    'ROSE_ChunkEval': 600,
    'HAT': 300,
    'ARF': 600,
    'SRP': 600,
    'ACDWM': 600,
    'ERulesD2S': 1800
}

# Controle de cache
USE_CACHE = True  # Se True, usa resultados existentes
FORCE_RERUN = []  # Lista de modelos para forcar re-execucao

# =============================================================================
# RESUMO DA CONFIGURACAO
# =============================================================================
print("=" * 70)
print("CONFIGURACAO DOS EXPERIMENTOS")
print("=" * 70)
print(f"\nUNIFIED_CHUNKS_DIR: {UNIFIED_CHUNKS_DIR}")
print(f"EXPERIMENTS_UNIFIED_DIR: {EXPERIMENTS_UNIFIED_DIR}")
print(f"\nExperimentos a executar: {EXPERIMENTS_TO_RUN}")
print(f"Modelos a executar: {MODELS_TO_RUN}")
print(f"Usar cache: {USE_CACHE}")
print(f"\nDatasets multiclasse (CDCMS/ACDWM N/A): {len(MULTICLASS_DATASETS)}")
print("=" * 70)


CONFIGURACAO DOS EXPERIMENTOS

UNIFIED_CHUNKS_DIR: /content/drive/Othercomputers/Laptop-CIn/Downloads/DSL-AG-hybrid/unified_chunks
EXPERIMENTS_UNIFIED_DIR: /content/drive/Othercomputers/Laptop-CIn/Downloads/DSL-AG-hybrid/experiments_unified

Experimentos a executar: ['exp_unified_500', 'exp_unified_500_penalty', 'exp_unified_1000', 'exp_unified_1000_penalty']
Modelos a executar: ['ROSE_Original', 'ROSE_ChunkEval', 'HAT', 'ARF', 'SRP', 'ACDWM', 'ERulesD2S']
Usar cache: False

Datasets multiclasse (CDCMS/ACDWM N/A): 9


In [ ]:
# =============================================================================
# CELULA 2.2: Auto-detectar Datasets Disponiveis (UNIFIED + LEGACY)
# =============================================================================
# SUBSTITUA a CELULA 2.2 original por este codigo
# Suporta deteccao de datasets em unified_chunks e diretorios legados
# =============================================================================

from pathlib import Path
from collections import defaultdict

def detect_available_datasets(exp_name, config):
    """
    Detecta quais datasets estao disponiveis para um experimento.

    Suporta dois modos:
    1. UNIFIED: Verifica unified_chunks + experiments_unified
    2. LEGACY: Verifica estrutura antiga (run_1/chunk_data)

    Returns:
        Lista de (batch_name, dataset_name, dataset_dir) para datasets disponiveis
    """
    available = []
    data_source = config.get('data_source', 'legacy')

    for batch_name, batch_info in config['batches'].items():
        base_dir = Path(WORK_DIR) / batch_info['base_dir']

        if not base_dir.exists():
            continue

        for dataset_name in batch_info['datasets']:
            dataset_dir = base_dir / dataset_name

            if data_source == 'unified':
                # MODO UNIFIED: Verificar dados em unified_chunks
                data_dir = config.get('data_dir', 'chunk_500')
                data_path = UNIFIED_CHUNKS_DIR / data_dir / dataset_name

                # Verificar se tem chunks de dados
                has_data = data_path.exists() and any(data_path.glob('chunk_*.csv'))

                # Verificar se tem resultados EGIS
                results_dir = config.get('results_dir', data_dir)
                egis_metrics = EXPERIMENTS_UNIFIED_DIR / results_dir / batch_name / dataset_name / 'run_1' / 'chunk_metrics.json'
                has_egis = egis_metrics.exists()

                if has_data and has_egis:
                    available.append((batch_name, dataset_name, dataset_dir))
                elif has_data and not has_egis:
                    # Dataset tem dados mas nao tem EGIS ainda
                    # Pode adicionar com flag indicando que EGIS esta faltando
                    pass

            else:
                # MODO LEGACY: Verificar estrutura antiga
                run_dir = dataset_dir / 'run_1'
                chunk_data_dir = run_dir / 'chunk_data'

                if chunk_data_dir.exists() and any(chunk_data_dir.glob('chunk_*_test.csv')):
                    available.append((batch_name, dataset_name, dataset_dir))

    return available


def detect_available_datasets_unified(exp_name, config):
    """
    Versao especifica para experimentos unified.
    Retorna informacoes adicionais sobre status de EGIS e CDCMS.

    Returns:
        Lista de dicts com informacoes detalhadas
    """
    available = []
    data_dir = config.get('data_dir', 'chunk_500')
    results_dir = config.get('results_dir', data_dir)

    for batch_name, batch_info in config['batches'].items():
        for dataset_name in batch_info['datasets']:
            # Verificar dados
            data_path = UNIFIED_CHUNKS_DIR / data_dir / dataset_name
            has_data = data_path.exists() and any(data_path.glob('chunk_*.csv'))

            if not has_data:
                continue

            # Verificar EGIS
            egis_path = EXPERIMENTS_UNIFIED_DIR / results_dir / batch_name / dataset_name / 'run_1' / 'chunk_metrics.json'
            has_egis = egis_path.exists()

            # Verificar CDCMS (sempre na pasta sem penalty)
            base_results_dir = results_dir.replace('_penalty', '')
            cdcms_path = EXPERIMENTS_UNIFIED_DIR / base_results_dir / batch_name / dataset_name / 'cdcms_results' / 'chunk_metrics.json'
            has_cdcms = cdcms_path.exists()

            # Verificar se e multiclasse
            is_multiclass = dataset_name in MULTICLASS_DATASETS

            available.append({
                'batch': batch_name,
                'dataset': dataset_name,
                'data_path': data_path,
                'has_egis': has_egis,
                'has_cdcms': has_cdcms,
                'is_multiclass': is_multiclass,
                'base_dir': batch_info['base_dir']
            })

    return available


# =============================================================================
# MOSTRAR RESUMO DOS DATASETS DISPONIVEIS
# =============================================================================
print("=" * 80)
print("DATASETS DISPONIVEIS POR EXPERIMENTO")
print("=" * 80)

for exp_name in EXPERIMENTS_TO_RUN:
    config = EXPERIMENT_CONFIGS[exp_name]
    data_source = config.get('data_source', 'legacy')

    print(f"\n{'='*60}")
    print(f"{exp_name}")
    print(f"  Fonte: {data_source}")
    print(f"  chunk_size: {config['chunk_size']}")
    print(f"  penalty_weight: {config.get('penalty_weight', 0.0)}")
    print(f"  {config['description']}")
    print(f"{'='*60}")

    if data_source == 'unified':
        # Usar deteccao detalhada para unified
        datasets_info = detect_available_datasets_unified(exp_name, config)

        # Agrupar por batch
        by_batch = defaultdict(list)
        for info in datasets_info:
            by_batch[info['batch']].append(info)

        total = 0
        egis_ok = 0
        cdcms_ok = 0
        multiclass = 0

        for batch, items in sorted(by_batch.items()):
            print(f"\n  {batch}: {len(items)} datasets")

            for item in items:
                egis_str = "EGIS:OK" if item['has_egis'] else "EGIS:--"
                cdcms_str = "CDCMS:OK" if item['has_cdcms'] else ("CDCMS:N/A" if item['is_multiclass'] else "CDCMS:--")
                mc_str = " [MULTICLASS]" if item['is_multiclass'] else ""

                print(f"    {item['dataset']}: {egis_str}, {cdcms_str}{mc_str}")

                total += 1
                if item['has_egis']:
                    egis_ok += 1
                if item['has_cdcms']:
                    cdcms_ok += 1
                if item['is_multiclass']:
                    multiclass += 1

        print(f"\n  RESUMO: {total} datasets, EGIS={egis_ok}, CDCMS={cdcms_ok}, Multiclass={multiclass}")

    else:
        # Usar deteccao simples para legacy
        available = detect_available_datasets(exp_name, config)

        by_batch = defaultdict(list)
        for batch, dataset, _ in available:
            by_batch[batch].append(dataset)

        for batch, datasets in sorted(by_batch.items()):
            print(f"  {batch}: {len(datasets)} datasets")

        print(f"\n  TOTAL: {len(available)} datasets")

print("\n" + "=" * 80)


DATASETS DISPONIVEIS POR EXPERIMENTO

exp_unified_500
  Fonte: unified
  chunk_size: 500
  penalty_weight: 0.0
  Unified chunks 500 (sem penalty)

  batch_1: 18 datasets
    SEA_Abrupt_Simple: EGIS:OK, CDCMS:OK
    SEA_Abrupt_Chain: EGIS:OK, CDCMS:OK
    SEA_Abrupt_Recurring: EGIS:OK, CDCMS:OK
    SEA_Gradual_Simple_Fast: EGIS:OK, CDCMS:OK
    SEA_Gradual_Simple_Slow: EGIS:OK, CDCMS:OK
    SEA_Gradual_Recurring: EGIS:OK, CDCMS:OK
    AGRAWAL_Abrupt_Simple_Mild: EGIS:OK, CDCMS:OK
    AGRAWAL_Abrupt_Simple_Severe: EGIS:OK, CDCMS:OK
    AGRAWAL_Abrupt_Chain_Long: EGIS:OK, CDCMS:OK
    HYPERPLANE_Abrupt_Simple: EGIS:OK, CDCMS:OK
    RANDOMTREE_Abrupt_Simple: EGIS:OK, CDCMS:OK
    RBF_Abrupt_Severe: EGIS:OK, CDCMS:OK
    RBF_Abrupt_Blip: EGIS:OK, CDCMS:OK
    RBF_Gradual_Moderate: EGIS:OK, CDCMS:OK
    RBF_Gradual_Severe: EGIS:OK, CDCMS:OK
    STAGGER_Abrupt_Chain: EGIS:OK, CDCMS:OK
    STAGGER_Abrupt_Recurring: EGIS:OK, CDCMS:OK
    STAGGER_Gradual_Chain: EGIS:OK, CDCMS:OK

  batch_2: 17 d

---
## PARTE 3: FUNÇÕES AUXILIARES
---

In [ ]:
# =============================================================================
# CELULA 3.1: Funcoes para Carregar Dados (UNIFIED + LEGACY)
# =============================================================================
# SUBSTITUA a CELULA 3.1 original por este codigo
# Adiciona funcoes para carregar dados de unified_chunks
# Adiciona funcoes para carregar resultados EGIS e CDCMS
# Mantem funcoes legadas para compatibilidade
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

# =============================================================================
# 1. FUNCOES PARA UNIFIED_CHUNKS (NOVAS)
# =============================================================================

def load_chunks_from_unified(data_dir, dataset_name):
    """
    Carrega chunks de dados do diretorio unified_chunks.
    Formato: unified_chunks/chunk_XXX/dataset/chunk_N.csv

    Args:
        data_dir: Nome do diretorio de dados (ex: 'chunk_500', 'chunk_1000')
        dataset_name: Nome do dataset (ex: 'SEA_Abrupt_Simple')

    Returns:
        X_chunks: Lista de arrays numpy (features)
        y_chunks: Lista de arrays numpy (labels)
    """
    data_path = UNIFIED_CHUNKS_DIR / data_dir / dataset_name

    if not data_path.exists():
        print(f"  [ERRO] Diretorio nao encontrado: {data_path}")
        return None, None

    X_chunks = []
    y_chunks = []

    # Ordenar chunks por numero
    chunk_files = sorted(
        data_path.glob("chunk_*.csv"),
        key=lambda x: int(x.stem.split('_')[1])
    )

    if not chunk_files:
        print(f"  [ERRO] Nenhum chunk encontrado em: {data_path}")
        return None, None

    for chunk_file in chunk_files:
        try:
            df = pd.read_csv(chunk_file)

            # Assumir ultima coluna como classe
            X = df.iloc[:, :-1].values.astype(float)
            y = df.iloc[:, -1].values

            # Converter classe para int
            if y.dtype == bool:
                y = y.astype(int)
            elif y.dtype == object:
                y = np.array([1 if str(v).lower() in ['true', '1', '1.0'] else 0 for v in y])
            elif 'float' in str(y.dtype):
                y = y.astype(int)

            X_chunks.append(X)
            y_chunks.append(y)

        except Exception as e:
            print(f"  [ERRO] Lendo {chunk_file.name}: {e}")
            continue

    if len(X_chunks) == 0:
        return None, None

    return X_chunks, y_chunks


def load_egis_results(results_dir, batch_name, dataset_name, egis_model_name='EGIS'):
    """
    Carrega resultados EGIS do chunk_metrics.json.

    Path: experiments_unified/{results_dir}/batch_Y/dataset/run_1/chunk_metrics.json

    Args:
        results_dir: Diretorio de resultados (ex: 'chunk_500', 'chunk_500_penalty')
        batch_name: Nome do batch (ex: 'batch_1')
        dataset_name: Nome do dataset
        egis_model_name: Nome do modelo EGIS ('EGIS' ou 'EGIS_Penalty')

    Returns:
        dict com 'gmean', 'f1', 'model_name' ou None se nao encontrado
    """
    run_dir = EXPERIMENTS_UNIFIED_DIR / results_dir / batch_name / dataset_name / "run_1"
    metrics_file = run_dir / "chunk_metrics.json"

    if not metrics_file.exists():
        return None

    try:
        with open(metrics_file) as f:
            chunk_metrics = json.load(f)

        # Calcular media dos test_gmean
        test_gmeans = [m.get('test_gmean', 0) for m in chunk_metrics if m.get('test_gmean') is not None]
        test_f1s = [m.get('test_f1', 0) for m in chunk_metrics if m.get('test_f1') is not None]
        test_accs = [m.get('test_accuracy', 0) for m in chunk_metrics if m.get('test_accuracy') is not None]

        if test_gmeans:
            return {
                'gmean': np.mean(test_gmeans),
                'f1': np.mean(test_f1s) if test_f1s else 0.0,
                'accuracy': np.mean(test_accs) if test_accs else 0.0,
                'num_chunks': len(chunk_metrics),
                'model_name': egis_model_name,
                'chunk_results': [
                    {
                        'chunk': i + 1,
                        'test_gmean': m.get('test_gmean', 0),
                        'test_accuracy': m.get('test_accuracy', 0),
                        'test_f1': m.get('test_f1', 0)
                    }
                    for i, m in enumerate(chunk_metrics)
                ]
            }
    except Exception as e:
        print(f"  [ERRO] Lendo EGIS results: {e}")

    return None


def load_cdcms_results(results_dir, batch_name, dataset_name):
    """
    Carrega resultados CDCMS do chunk_metrics.json.

    NOTA: CDCMS so existe nas pastas SEM penalty.
    Path: experiments_unified/chunk_XXX/batch_Y/dataset/cdcms_results/chunk_metrics.json

    Args:
        results_dir: Diretorio de resultados (automaticamente remove _penalty)
        batch_name: Nome do batch
        dataset_name: Nome do dataset

    Returns:
        dict com 'gmean', 'prequential_gmean', 'holdout_gmean' ou None
    """
    # CDCMS sempre na pasta sem penalty
    base_results_dir = results_dir.replace('_penalty', '')

    cdcms_dir = EXPERIMENTS_UNIFIED_DIR / base_results_dir / batch_name / dataset_name / "cdcms_results"
    metrics_file = cdcms_dir / "chunk_metrics.json"

    if not metrics_file.exists():
        return None

    try:
        with open(metrics_file) as f:
            chunk_metrics = json.load(f)

        # Calcular medias
        prequential_gmeans = [m.get('prequential_gmean', 0) for m in chunk_metrics if m.get('prequential_gmean') is not None]
        holdout_gmeans = [m.get('holdout_gmean') for m in chunk_metrics if m.get('holdout_gmean') is not None]

        # Preferir holdout_gmean se disponivel
        if holdout_gmeans:
            gmean = np.mean(holdout_gmeans)
        elif prequential_gmeans:
            gmean = np.mean(prequential_gmeans)
        else:
            gmean = 0.0

        return {
            'gmean': gmean,
            'prequential_gmean': np.mean(prequential_gmeans) if prequential_gmeans else 0.0,
            'holdout_gmean': np.mean(holdout_gmeans) if holdout_gmeans else None,
            'num_chunks': len(chunk_metrics),
            'chunk_results': [
                {
                    'chunk': i + 1,
                    'prequential_gmean': m.get('prequential_gmean', 0),
                    'holdout_gmean': m.get('holdout_gmean', 0)
                }
                for i, m in enumerate(chunk_metrics)
            ]
        }
    except Exception as e:
        print(f"  [ERRO] Lendo CDCMS results: {e}")

    return None


def load_existing_comparative_results(results_dir, batch_name, dataset_name, model_name):
    """
    Carrega resultados existentes de um modelo comparativo em experimentos unified.

    Os modelos comparativos sao salvos nas pastas SEM penalty.

    Args:
        results_dir: Diretorio de resultados
        batch_name: Nome do batch
        dataset_name: Nome do dataset
        model_name: Nome do modelo (HAT, ARF, SRP, ROSE_Original, etc.)

    Returns:
        dict com 'gmean', 'status' ou None se nao encontrado
    """
    # Modelos comparativos sempre na pasta sem penalty
    base_results_dir = results_dir.replace('_penalty', '')
    dataset_dir = EXPERIMENTS_UNIFIED_DIR / base_results_dir / batch_name / dataset_name
    run_dir = dataset_dir / "run_1"

    # Mapear nome do modelo para arquivo
    file_map = {
        'HAT': 'river_HAT_results.csv',
        'ARF': 'river_ARF_results.csv',
        'SRP': 'river_SRP_results.csv',
        'ACDWM': 'acdwm_results.csv',
        'ERulesD2S': 'erulesd2s_results.csv',
        'ROSE_Original': 'rose_original_results.csv',
        'ROSE_ChunkEval': 'rose_chunk_eval_results.csv'
    }

    if model_name not in file_map:
        return None

    result_file = run_dir / file_map[model_name]

    if not result_file.exists():
        return None

    try:
        df = pd.read_csv(result_file)

        # Procurar coluna de gmean
        for col in ['test_gmean', 'gmean', 'G-Mean', 'g_mean']:
            if col in df.columns:
                return {'gmean': df[col].mean(), 'status': 'CACHED'}

        # Fallback: accuracy
        if 'accuracy' in df.columns or 'test_accuracy' in df.columns:
            col = 'test_accuracy' if 'test_accuracy' in df.columns else 'accuracy'
            return {'gmean': df[col].mean(), 'status': 'CACHED_ACC'}

    except Exception as e:
        pass

    return None


# =============================================================================
# 2. FUNCOES LEGADAS (MANTIDAS PARA COMPATIBILIDADE)
# =============================================================================

def load_chunks_from_gbml(dataset_dir):
    """
    [LEGADO] Carrega chunks de dados do diretorio GBML antigo.
    Formato: chunk_data/chunk_X_test.csv (X = 1, 2, 3, ...)
    """
    dataset_dir = Path(dataset_dir)
    run_dir = dataset_dir / "run_1"
    chunk_data_dir = run_dir / "chunk_data"

    X_chunks = []
    y_chunks = []

    if not chunk_data_dir.exists():
        return None, None

    # Procurar chunks de teste
    test_files = sorted(chunk_data_dir.glob("chunk_*_test.csv"))

    for test_file in test_files:
        try:
            df = pd.read_csv(test_file)

            # Identificar coluna de classe
            if 'class' in df.columns:
                y = df['class'].values
                X = df.drop(columns=['class']).values
            elif 'label' in df.columns:
                y = df['label'].values
                X = df.drop(columns=['label']).values
            else:
                y = df.iloc[:, -1].values
                X = df.iloc[:, :-1].values

            X_chunks.append(X.astype(float))
            y_chunks.append(y.astype(int))
        except Exception as e:
            print(f"  Erro ao carregar {test_file}: {e}")
            continue

    if len(X_chunks) > 0:
        return X_chunks, y_chunks
    return None, None


def load_existing_model_results(dataset_dir, model_name):
    """
    [LEGADO] Carrega resultados existentes de um modelo comparativo.
    Retorna dict com gmean ou None.
    """
    dataset_dir = Path(dataset_dir)
    run_dir = dataset_dir / "run_1"

    # Mapear nome do modelo para arquivo
    file_map = {
        'HAT': 'river_HAT_results.csv',
        'ARF': 'river_ARF_results.csv',
        'SRP': 'river_SRP_results.csv',
        'ACDWM': 'acdwm_results.csv',
        'ERulesD2S': 'erulesd2s_results.csv',
        'ROSE_Original': 'rose_original_results.csv',
        'ROSE_ChunkEval': 'rose_chunk_eval_results.csv'
    }

    if model_name not in file_map:
        return None

    result_file = run_dir / file_map[model_name]

    if result_file.exists():
        try:
            df = pd.read_csv(result_file)

            # Procurar coluna de gmean
            for col in ['test_gmean', 'gmean', 'G-Mean', 'g_mean']:
                if col in df.columns:
                    return {'gmean': df[col].mean(), 'source': 'cached'}

            # Fallback: accuracy
            if 'accuracy' in df.columns or 'test_accuracy' in df.columns:
                col = 'test_accuracy' if 'test_accuracy' in df.columns else 'accuracy'
                return {'gmean': df[col].mean(), 'source': 'cached_accuracy'}

        except Exception as e:
            pass

    return None


# =============================================================================
# 3. FUNCOES AUXILIARES
# =============================================================================

def get_dataset_output_dir(exp_config, batch_name, dataset_name):
    """
    Retorna o diretorio onde salvar resultados de modelos comparativos.
    Para experimentos com penalty, salva na pasta sem penalty.
    """
    data_source = exp_config.get('data_source', 'legacy')

    if data_source == 'unified':
        # Modelos comparativos sempre na pasta sem penalty
        results_dir = exp_config.get('results_dir', 'chunk_500').replace('_penalty', '')
        return EXPERIMENTS_UNIFIED_DIR / results_dir / batch_name / dataset_name
    else:
        # Legacy: usar base_dir do batch
        batch_info = exp_config['batches'].get(batch_name, {})
        return Path(WORK_DIR) / batch_info.get('base_dir', '') / dataset_name


# =============================================================================
# TESTE RAPIDO DAS FUNCOES
# =============================================================================
print("=" * 70)
print("CELULA 3.1: FUNCOES DE CARREGAMENTO")
print("=" * 70)

# Testar se diretorios existem
print(f"\nVerificando diretorios:")
print(f"  UNIFIED_CHUNKS_DIR: {UNIFIED_CHUNKS_DIR.exists()}")
print(f"  EXPERIMENTS_UNIFIED_DIR: {EXPERIMENTS_UNIFIED_DIR.exists()}")

# Testar carregamento de um dataset unified
test_data_dir = 'chunk_500'
test_dataset = 'SEA_Abrupt_Simple'

if UNIFIED_CHUNKS_DIR.exists():
    X_test, y_test = load_chunks_from_unified(test_data_dir, test_dataset)
    if X_test is not None:
        print(f"\n  Teste load_chunks_from_unified:")
        print(f"    Dataset: {test_data_dir}/{test_dataset}")
        print(f"    Chunks: {len(X_test)}, Samples/chunk: {len(X_test[0])}, Features: {X_test[0].shape[1]}")
        print(f"    Classes: {np.unique(y_test[0])}")

# Testar carregamento de EGIS
if EXPERIMENTS_UNIFIED_DIR.exists():
    egis = load_egis_results('chunk_500', 'batch_1', test_dataset, 'EGIS')
    if egis:
        print(f"\n  Teste load_egis_results:")
        print(f"    Model: {egis['model_name']}")
        print(f"    G-Mean: {egis['gmean']:.4f}")
        print(f"    Chunks: {egis['num_chunks']}")

    cdcms = load_cdcms_results('chunk_500', 'batch_1', test_dataset)
    if cdcms:
        print(f"\n  Teste load_cdcms_results:")
        print(f"    G-Mean: {cdcms['gmean']:.4f}")
        print(f"    Chunks: {cdcms['num_chunks']}")

print("\n" + "=" * 70)
print("FUNCOES CARREGADAS:")
print("=" * 70)
print("  UNIFIED:")
print("    - load_chunks_from_unified(data_dir, dataset_name)")
print("    - load_egis_results(results_dir, batch_name, dataset_name)")
print("    - load_cdcms_results(results_dir, batch_name, dataset_name)")
print("    - load_existing_comparative_results(results_dir, batch_name, dataset_name, model_name)")
print("    - get_dataset_output_dir(exp_config, batch_name, dataset_name)")
print("  LEGACY:")
print("    - load_chunks_from_gbml(dataset_dir)")
print("    - load_existing_model_results(dataset_dir, model_name)")
print("=" * 70)


CELULA 3.1: FUNCOES DE CARREGAMENTO

Verificando diretorios:
  UNIFIED_CHUNKS_DIR: True
  EXPERIMENTS_UNIFIED_DIR: True

  Teste load_chunks_from_unified:
    Dataset: chunk_500/SEA_Abrupt_Simple
    Chunks: 24, Samples/chunk: 500, Features: 3
    Classes: [0 1]

  Teste load_egis_results:
    Model: EGIS
    G-Mean: 0.9496
    Chunks: 23

  Teste load_cdcms_results:
    G-Mean: 0.9412
    Chunks: 24

FUNCOES CARREGADAS:
  UNIFIED:
    - load_chunks_from_unified(data_dir, dataset_name)
    - load_egis_results(results_dir, batch_name, dataset_name)
    - load_cdcms_results(results_dir, batch_name, dataset_name)
    - load_existing_comparative_results(results_dir, batch_name, dataset_name, model_name)
    - get_dataset_output_dir(exp_config, batch_name, dataset_name)
  LEGACY:
    - load_chunks_from_gbml(dataset_dir)
    - load_existing_model_results(dataset_dir, model_name)


In [ ]:
# CÉLULA 3.2: Funções para ARFF e ROSE

def get_rose_evaluator(n_classes):
    """Retorna o evaluator correto baseado no número de classes."""
    if n_classes <= 2:
        return "WindowAUCImbalancedPerformanceEvaluator"
    else:
        return "WindowAUCMultiClassImbalancedPerformanceEvaluator"


def create_arff_file(X, y, output_path, relation_name="stream"):
    """Cria arquivo ARFF a partir de dados numpy."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    n_samples, n_features = X.shape
    classes = sorted(np.unique(y))

    with open(output_path, 'w') as f:
        f.write(f"@relation {relation_name}\n\n")
        for i in range(n_features):
            f.write(f"@attribute attr{i} numeric\n")
        class_str = ",".join([str(c) for c in classes])
        f.write(f"@attribute class {{{class_str}}}\n\n")
        f.write("@data\n")
        for i in range(n_samples):
            row = ",".join([str(x) for x in X[i]]) + f",{y[i]}\n"
            f.write(row)

    return output_path


def run_rose_original(arff_file, output_dir, n_classes=2, timeout=600):
    """ROSE_Original: Executa ROSE como no paper original (métrica global)."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = output_dir / "rose_original_results.csv"
    log_file = output_dir / "rose_original_log.txt"

    classpath = "rose_jars/ROSE-1.0.jar:rose_jars/MOA-dependencies.jar"
    evaluator = get_rose_evaluator(n_classes)

    cmd = [
        "java", "-Xmx4g",
        f"-javaagent:rose_jars/sizeofag-1.0.4.jar",
        "-cp", classpath, "moa.DoTask"
    ]

    task_parts = [
        "EvaluateInterleavedTestThenTrain",
        "-e", f"({evaluator})",
        "-s", f"(ArffFileStream -f {arff_file})",
        "-l", "(moa.classifiers.meta.imbalanced.ROSE)",
        "-f", "500",
        "-d", str(output_file)
    ]

    task_string = " ".join(task_parts)
    cmd.append(task_string)

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)

        with open(log_file, 'w') as f:
            f.write(f"Command: {' '.join(cmd)}\n\n")
            f.write(f"STDOUT:\n{result.stdout}\n\nSTDERR:\n{result.stderr}")

        if result.returncode != 0:
            return False, {'gmean': 0.0, 'error': 'Non-zero return code'}

        results = parse_rose_results_global(output_file)
        return True, results

    except subprocess.TimeoutExpired:
        return False, {'gmean': 0.0, 'error': 'Timeout'}
    except Exception as e:
        return False, {'gmean': 0.0, 'error': str(e)}


def run_rose_chunk_eval(arff_file, output_dir, n_classes=2, chunk_size=1000, timeout=600):
    """ROSE_ChunkEval: Executa ROSE com avaliação por chunk (média)."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = output_dir / "rose_chunk_eval_results.csv"
    log_file = output_dir / "rose_chunk_eval_log.txt"

    classpath = "rose_jars/ROSE-1.0.jar:rose_jars/MOA-dependencies.jar"
    evaluator = get_rose_evaluator(n_classes)

    cmd = [
        "java", "-Xmx4g",
        f"-javaagent:rose_jars/sizeofag-1.0.4.jar",
        "-cp", classpath, "moa.DoTask"
    ]

    task_parts = [
        "EvaluateInterleavedTestThenTrain",
        "-e", f"({evaluator})",
        "-s", f"(ArffFileStream -f {arff_file})",
        "-l", "(moa.classifiers.meta.imbalanced.ROSE)",
        "-f", str(chunk_size),
        "-d", str(output_file)
    ]

    task_string = " ".join(task_parts)
    cmd.append(task_string)

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)

        with open(log_file, 'w') as f:
            f.write(f"Command: {' '.join(cmd)}\n\n")
            f.write(f"STDOUT:\n{result.stdout}\n\nSTDERR:\n{result.stderr}")

        if result.returncode != 0:
            return False, {'gmean': 0.0, 'error': 'Non-zero return code'}

        results = parse_rose_results_chunk_avg(output_file)
        return True, results

    except subprocess.TimeoutExpired:
        return False, {'gmean': 0.0, 'error': 'Timeout'}
    except Exception as e:
        return False, {'gmean': 0.0, 'error': str(e)}


def parse_rose_results_global(results_file):
    """Parseia resultados do ROSE - retorna métrica GLOBAL (última linha)."""
    results = {'gmean': 0.0, 'accuracy': 0.0}
    results_file = Path(results_file)

    if not results_file.exists():
        return results

    try:
        with open(results_file, 'r') as f:
            lines = f.readlines()

        data_lines = [line for line in lines
                      if not line.startswith('Learner')
                      and not line.startswith('learning')
                      and line.strip()]

        if len(data_lines) > 0:
            header_line = lines[0]
            last_data_line = data_lines[-1]

            headers = header_line.strip().split(',')
            values = last_data_line.strip().split(',')
            data_dict = dict(zip(headers, values))

            if 'G-Mean' in data_dict:
                try:
                    results['gmean'] = float(data_dict['G-Mean'])
                except:
                    pass
            if 'Accuracy' in data_dict:
                try:
                    results['accuracy'] = float(data_dict['Accuracy'])
                except:
                    pass

    except Exception as e:
        results['error'] = str(e)

    return results


def parse_rose_results_chunk_avg(results_file):
    """Parseia resultados do ROSE - retorna MÉDIA das métricas por chunk."""
    results = {'gmean': 0.0, 'accuracy': 0.0}
    results_file = Path(results_file)

    if not results_file.exists():
        return results

    try:
        with open(results_file, 'r') as f:
            lines = f.readlines()

        data_lines = [line for line in lines
                      if not line.startswith('Learner')
                      and not line.startswith('learning')
                      and line.strip()]

        if len(data_lines) > 0:
            header_line = lines[0]
            headers = header_line.strip().split(',')

            gmeans = []
            accuracies = []

            for data_line in data_lines:
                values = data_line.strip().split(',')
                data_dict = dict(zip(headers, values))

                if 'G-Mean' in data_dict:
                    try:
                        gmeans.append(float(data_dict['G-Mean']))
                    except:
                        pass
                if 'Accuracy' in data_dict:
                    try:
                        accuracies.append(float(data_dict['Accuracy']))
                    except:
                        pass

            if gmeans:
                results['gmean'] = np.mean(gmeans)
            if accuracies:
                results['accuracy'] = np.mean(accuracies)
            results['n_chunks_evaluated'] = len(gmeans)

    except Exception as e:
        results['error'] = str(e)

    return results


print("Funções ROSE definidas!")

Funções ROSE definidas!


In [ ]:
# CÉLULA 3.3: Funções para River Models (HAT, ARF, SRP)

from river import tree, ensemble, forest
from sklearn.metrics import accuracy_score, f1_score

def calculate_gmean(y_true, y_pred):
    """Calcula G-mean."""
    from sklearn.metrics import confusion_matrix
    classes = np.unique(y_true)

    if len(classes) == 2:
        cm = confusion_matrix(y_true, y_pred, labels=classes)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
            return np.sqrt(sensitivity * specificity)

    # Multi-class
    recalls = []
    for c in classes:
        mask = y_true == c
        if mask.sum() > 0:
            recall = np.mean(y_pred[mask] == c)
            recalls.append(recall)

    if len(recalls) > 0:
        return np.prod(recalls) ** (1.0 / len(recalls))
    return 0.0


def run_river_model(model_name, X_chunks, y_chunks, timeout=300):
    """
    Executa modelo River (HAT, ARF, SRP) nos chunks.
    Retorna métricas por chunk e global.
    """
    try:
        # Criar modelo
        if model_name == 'HAT':
            model = tree.HoeffdingAdaptiveTreeClassifier()
        elif model_name == 'ARF':
            # NOTA: Em River >= 0.21, ARF foi movido de ensemble para forest
            model = forest.ARFClassifier(n_models=10)
        elif model_name == 'SRP':
            model = ensemble.SRPClassifier(n_models=10)
        else:
            return {'gmean': 0.0, 'error': f'Unknown model: {model_name}'}

        all_preds = []
        all_true = []
        chunk_results = []

        for chunk_idx, (X, y) in enumerate(zip(X_chunks, y_chunks)):
            chunk_preds = []

            for i in range(len(X)):
                x_dict = {f'f{j}': float(X[i, j]) for j in range(X.shape[1])}
                y_i = int(y[i])

                # Predict
                pred = model.predict_one(x_dict)
                if pred is None:
                    pred = 0
                chunk_preds.append(pred)

                # Learn
                model.learn_one(x_dict, y_i)

            all_preds.extend(chunk_preds)
            all_true.extend(y)

            # Chunk metrics
            chunk_gmean = calculate_gmean(np.array(y), np.array(chunk_preds))
            chunk_acc = accuracy_score(y, chunk_preds)
            chunk_results.append({
                'chunk': chunk_idx + 1,
                'test_gmean': chunk_gmean,
                'test_accuracy': chunk_acc
            })

        # Global metrics
        all_preds = np.array(all_preds)
        all_true = np.array(all_true)

        gmean = calculate_gmean(all_true, all_preds)
        accuracy = accuracy_score(all_true, all_preds)

        return {
            'gmean': gmean,
            'accuracy': accuracy,
            'chunk_results': chunk_results
        }

    except Exception as e:
        return {'gmean': 0.0, 'accuracy': 0.0, 'error': str(e)}


print("Funções River definidas!")

Funções River definidas!


In [ ]:
# CÉLULA 3.4: Funções para ACDWM

def run_acdwm(X_chunks, y_chunks, acdwm_path="/content/ACDWM", timeout=600):
    """
    Executa ACDWM nos chunks.
    LIMITAÇÃO: ACDWM só funciona com problemas BINÁRIOS (2 classes).
    """
    try:
        import sys
        if acdwm_path not in sys.path:
            sys.path.insert(0, acdwm_path)

        # Verificar número de classes
        all_y = np.concatenate(y_chunks)
        unique_classes = np.unique(all_y)
        n_classes = len(unique_classes)

        if n_classes > 2:
            return {
                'gmean': 0.0,
                'accuracy': 0.0,
                'error': f'ACDWM does not support multiclass (n_classes={n_classes})'
            }

        if n_classes < 2:
            return {
                'gmean': 0.0,
                'accuracy': 0.0,
                'error': f'Need at least 2 classes (found {n_classes})'
            }

        from dwmil import DWMIL

        model = DWMIL(
            data_num=999999,
            chunk_size=0,
            theta=0.001,
            err_func='gm',
            r=1.0
        )

        all_preds = []
        all_true = []
        chunk_results = []

        def to_acdwm_labels(y):
            return np.where(y == 0, -1, 1).astype(np.int32)

        def from_acdwm_labels(y):
            return np.where(y == -1, 0, 1).astype(np.int32)

        for chunk_idx, (X, y) in enumerate(zip(X_chunks, y_chunks)):
            X = np.array(X, dtype=float)
            y = np.array(y, dtype=int)
            y_acdwm = to_acdwm_labels(y)

            try:
                y_pred_acdwm = model.update_chunk(X, y_acdwm)
                y_pred = from_acdwm_labels(y_pred_acdwm)

                all_preds.extend(y_pred)
                all_true.extend(y)

                # Chunk metrics
                chunk_gmean = calculate_gmean(y, y_pred)
                chunk_acc = accuracy_score(y, y_pred)
                chunk_results.append({
                    'chunk': chunk_idx + 1,
                    'test_gmean': chunk_gmean,
                    'test_accuracy': chunk_acc
                })

            except Exception as e:
                return {
                    'gmean': 0.0,
                    'accuracy': 0.0,
                    'error': f'ACDWM failed on chunk {chunk_idx}: {str(e)[:50]}'
                }

        if len(all_preds) == 0:
            return {'gmean': 0.0, 'accuracy': 0.0, 'error': 'No predictions made'}

        all_preds = np.array(all_preds)
        all_true = np.array(all_true)

        gmean = calculate_gmean(all_true, all_preds)
        accuracy = accuracy_score(all_true, all_preds)

        return {
            'gmean': gmean,
            'accuracy': accuracy,
            'chunk_results': chunk_results
        }

    except ImportError as e:
        return {
            'gmean': 0.0,
            'accuracy': 0.0,
            'error': f'Could not import DWMIL: {str(e)[:50]}'
        }
    except Exception as e:
        return {
            'gmean': 0.0,
            'accuracy': 0.0,
            'error': f'ACDWM error: {str(e)[:50]}'
        }


print("Funções ACDWM definidas!")

Funções ACDWM definidas!


In [ ]:
# CÉLULA 3.5: Funções para salvar resultados

def save_model_results(dataset_dir, model_name, results):
    """
    Salva resultados de um modelo no diretório do dataset.
    Formato: run_1/river_<model>_results.csv ou run_1/<model>_results.csv
    """
    dataset_dir = Path(dataset_dir)
    run_dir = dataset_dir / "run_1"
    run_dir.mkdir(parents=True, exist_ok=True)

    # Nome do arquivo de saída
    if model_name in ['HAT', 'ARF', 'SRP']:
        output_file = run_dir / f"river_{model_name}_results.csv"
    elif model_name == 'ACDWM':
        output_file = run_dir / "acdwm_results.csv"
    elif model_name == 'ROSE_Original':
        output_file = run_dir / "rose_original_results.csv"
    elif model_name == 'ROSE_ChunkEval':
        output_file = run_dir / "rose_chunk_eval_results.csv"
    else:
        output_file = run_dir / f"{model_name.lower()}_results.csv"

    # Se temos resultados por chunk, salvar detalhado
    if 'chunk_results' in results and results['chunk_results']:
        df = pd.DataFrame(results['chunk_results'])
        df.to_csv(output_file, index=False)
    else:
        # Salvar resultado global
        df = pd.DataFrame([{
            'chunk': 1,
            'test_gmean': results.get('gmean', 0.0),
            'test_accuracy': results.get('accuracy', 0.0)
        }])
        df.to_csv(output_file, index=False)

    return output_file


print("Funções de salvamento definidas!")

Funções de salvamento definidas!


In [ ]:
# CÉLULA 3.6: Funções para ERulesD2S (EXECUÇÃO REAL)

# Flag para controlar execução do ERulesD2S
ERULESD2S_ENABLED = True  # Mude para False se quiser apenas usar cache
ERULESD2S_JAR = Path(WORK_DIR) / "erulesd2s.jar"
ERULESD2S_JCLEC_JAR = Path(WORK_DIR) / "lib" / "JCLEC4-base-1.0-jar-with-dependencies.jar"

def run_erulesd2s(X_chunks, y_chunks, dataset_dir, dataset_name, chunk_size=1000, timeout=600):
    """
    Executa ERulesD2S nos chunks usando o wrapper Java/MOA.

    IMPORTANTE: Baseado no erulesd2s_wrapper.py que funciona corretamente.

    Args:
        X_chunks: Lista de arrays X (features)
        y_chunks: Lista de arrays y (labels)
        dataset_dir: Diretório do dataset
        dataset_name: Nome do dataset
        chunk_size: Tamanho do chunk para avaliação
        timeout: Timeout em segundos (padrão: 10 min - ajustado para velocidade)

    Returns:
        dict com 'gmean', 'accuracy', 'chunk_results' ou 'error'
    """
    import shlex

    # Verificar se JAR existe
    if not ERULESD2S_JAR.exists():
        return {'gmean': 0.0, 'error': f'erulesd2s.jar not found at {ERULESD2S_JAR}'}

    dataset_dir = Path(dataset_dir)
    run_dir = dataset_dir / "run_1"
    run_dir.mkdir(parents=True, exist_ok=True)

    # Concatenar todos os chunks em um único dataset
    X_all = np.vstack(X_chunks)
    y_all = np.concatenate(y_chunks)

    # Criar arquivo ARFF
    arff_dir = run_dir / "erulesd2s_arff"
    arff_dir.mkdir(parents=True, exist_ok=True)
    arff_file = arff_dir / f"{dataset_name}.arff"
    create_arff_file(X_all, y_all, arff_file, relation_name=dataset_name)

    # Diretório de saída
    output_dir = run_dir / "erulesd2s_run"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / "erulesd2s_output.csv"
    log_file = output_dir / "erulesd2s_log.txt"

    # Construir classpath (Linux usa :)
    classpath_parts = [str(ERULESD2S_JAR)]
    if ERULESD2S_JCLEC_JAR.exists():
        classpath_parts.append(str(ERULESD2S_JCLEC_JAR))

    # Adicionar outras JARs na pasta lib/
    lib_dir = Path(WORK_DIR) / "lib"
    if lib_dir.exists():
        for jar in lib_dir.glob("*.jar"):
            if str(jar) not in classpath_parts:
                classpath_parts.append(str(jar))

    classpath = ":".join(classpath_parts)

    # Parâmetros ERulesD2S (REDUZIDOS para velocidade)
    population_size = 25   # Reduzido de 25
    num_generations = 50   # Reduzido de 50
    rules_per_class = 5    # Reduzido de 5

    # Construir comando como lista (formato correto para subprocess)
    # IMPORTANTE: A task string precisa ser um único argumento para moa.DoTask
    learner = f"(moa.classifiers.evolutionary.EvolutionaryRuleLearner -s {population_size} -g {num_generations} -r {rules_per_class})"
    stream = f"(ArffFileStream -f {arff_file})"

    task_string = f"EvaluateInterleavedTestThenTrain -s {stream} -l {learner} -f {chunk_size} -d {output_file}"

    cmd = [
        "java",
        "-Xmx4g",
        "-cp", classpath,
        "moa.DoTask",
        task_string
    ]

    try:
        print(f"    Executando ERulesD2S (timeout={timeout}s)...")
        start_time = time.time()

        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=timeout,
            cwd=str(Path(WORK_DIR))
        )

        duration = time.time() - start_time

        # Salvar log
        with open(log_file, 'w') as f:
            f.write(f"Command: {' '.join(cmd)}\n\n")
            f.write(f"Duration: {duration:.1f}s\n\n")
            f.write(f"Return code: {result.returncode}\n\n")
            f.write(f"STDOUT:\n{result.stdout}\n\n")
            f.write(f"STDERR:\n{result.stderr}\n")

        if result.returncode != 0:
            error_msg = result.stderr[:200] if result.stderr else "Unknown error"
            return {
                'gmean': 0.0,
                'error': f'returncode={result.returncode}: {error_msg}'
            }

        # Parsear resultados
        if output_file.exists():
            try:
                with open(output_file, 'r') as f:
                    lines = f.readlines()

                # Filtrar linhas de dados (não headers)
                data_lines = [line for line in lines
                              if not line.startswith('Learner')
                              and not line.startswith('learning')
                              and line.strip()]

                if data_lines:
                    header_line = lines[0]
                    headers = header_line.strip().split(',')

                    accuracies = []
                    chunk_results = []

                    for chunk_idx, data_line in enumerate(data_lines):
                        values = data_line.strip().split(',')
                        data_dict = dict(zip(headers, values))

                        # Extrair accuracy
                        acc = 0.0
                        if 'classifications correct (percent)' in data_dict:
                            try:
                                acc = float(data_dict['classifications correct (percent)']) / 100.0
                            except:
                                pass

                        accuracies.append(acc)
                        chunk_results.append({
                            'chunk': chunk_idx + 1,
                            'test_gmean': acc,
                            'test_accuracy': acc
                        })

                    # Calcular média
                    gmean = np.mean(accuracies) if accuracies else 0.0

                    # Salvar resultados no formato esperado
                    results_file = run_dir / "erulesd2s_results.csv"
                    df = pd.DataFrame(chunk_results)
                    df['model'] = 'ERulesD2S'
                    df['execution_time'] = duration / len(chunk_results) if chunk_results else 0
                    df.to_csv(results_file, index=False)

                    print(f"    ERulesD2S concluído em {duration:.1f}s (gmean={gmean:.4f})")

                    return {
                        'gmean': gmean,
                        'accuracy': gmean,
                        'chunk_results': chunk_results,
                        'execution_time': duration
                    }

            except Exception as e:
                return {'gmean': 0.0, 'error': f'Error parsing: {str(e)[:50]}'}

        return {'gmean': 0.0, 'error': 'No output file'}

    except subprocess.TimeoutExpired:
        print(f"    ERulesD2S TIMEOUT após {timeout}s")
        return {'gmean': 0.0, 'error': f'Timeout ({timeout}s)'}
    except Exception as e:
        return {'gmean': 0.0, 'error': f'Exception: {str(e)[:50]}'}


# Verificar se ERulesD2S está disponível
print("Verificação ERulesD2S:")
print(f"  erulesd2s.jar: {'OK' if ERULESD2S_JAR.exists() else 'FALTANDO'}")
print(f"  JCLEC4 JAR: {'OK' if ERULESD2S_JCLEC_JAR.exists() else 'FALTANDO'}")
print(f"  ERULESD2S_ENABLED: {ERULESD2S_ENABLED}")

if not ERULESD2S_JAR.exists():
    print("\n  AVISO: ERulesD2S não será executado (JAR não encontrado)")
    print("  Apenas resultados em cache serão usados")
    ERULESD2S_ENABLED = False

print("\nFunções ERulesD2S definidas!")

Verificação ERulesD2S:
  erulesd2s.jar: OK
  JCLEC4 JAR: OK
  ERULESD2S_ENABLED: True

Funções ERulesD2S definidas!


In [ ]:
# =============================================================================
# CELULAS 3.3, 3.4, 3.5, 3.6: METRICAS COMPLETAS (Consistente com CDCMS)
# =============================================================================
# SUBSTITUA as CELULAs 3.3, 3.4, 3.5 e 3.6 por este codigo
# Adiciona f1 e f1_weighted para TODOS os modelos
# Corrige ERulesD2S para calcular gmean corretamente
# =============================================================================
#
# METRICAS SALVAS POR MODELO (PADRAO CDCMS):
# - test_gmean
# - test_accuracy
# - test_f1 (macro)
# - test_f1_weighted
#
# =============================================================================

from river import tree, ensemble, forest
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import numpy as np
import pandas as pd
from pathlib import Path

# =============================================================================
# FUNCOES AUXILIARES DE METRICAS
# =============================================================================

def calculate_gmean(y_true, y_pred):
    """
    Calcula G-mean (media geometrica das recalls por classe).
    Funciona para binario e multiclasse.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    classes = np.unique(y_true)

    if len(classes) == 2:
        # Binario: sqrt(sensitivity * specificity)
        cm = confusion_matrix(y_true, y_pred, labels=classes)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
            return np.sqrt(sensitivity * specificity)

    # Multi-classe: raiz n-esima do produto das recalls
    recalls = []
    for c in classes:
        mask = y_true == c
        if mask.sum() > 0:
            recall = np.mean(y_pred[mask] == c)
            recalls.append(recall)

    if len(recalls) > 0:
        return np.prod(recalls) ** (1.0 / len(recalls))
    return 0.0


def calculate_f1_weighted(y_true, y_pred):
    """Calcula F1-weighted (considera desbalanceamento de classes)."""
    try:
        return f1_score(y_true, y_pred, average='weighted', zero_division=0)
    except:
        return 0.0


def calculate_f1_macro(y_true, y_pred):
    """Calcula F1-macro (media simples entre classes)."""
    try:
        return f1_score(y_true, y_pred, average='macro', zero_division=0)
    except:
        return 0.0


def calculate_all_metrics(y_true, y_pred):
    """
    Calcula todas as metricas padrao.
    Returns dict com gmean, accuracy, f1, f1_weighted.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    return {
        'gmean': calculate_gmean(y_true, y_pred),
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': calculate_f1_macro(y_true, y_pred),
        'f1_weighted': calculate_f1_weighted(y_true, y_pred)
    }


print("Funcoes auxiliares de metricas definidas!")

# =============================================================================
# CELULA 3.3: Funcoes para River Models (HAT, ARF, SRP) - METRICAS COMPLETAS
# =============================================================================

def run_river_model(model_name, X_chunks, y_chunks, timeout=300):
    """
    Executa modelo River (HAT, ARF, SRP) nos chunks.

    METRICAS SALVAS (consistente com CDCMS):
    - test_gmean
    - test_accuracy
    - test_f1 (macro)
    - test_f1_weighted
    """
    try:
        # Criar modelo
        if model_name == 'HAT':
            model = tree.HoeffdingAdaptiveTreeClassifier()
        elif model_name == 'ARF':
            model = forest.ARFClassifier(n_models=10)
        elif model_name == 'SRP':
            model = ensemble.SRPClassifier(n_models=10)
        else:
            return {'gmean': 0.0, 'error': f'Unknown model: {model_name}'}

        all_preds = []
        all_true = []
        chunk_results = []

        for chunk_idx, (X, y) in enumerate(zip(X_chunks, y_chunks)):
            chunk_preds = []

            for i in range(len(X)):
                x_dict = {f'f{j}': float(X[i, j]) for j in range(X.shape[1])}
                y_i = int(y[i])

                # Predict
                pred = model.predict_one(x_dict)
                if pred is None:
                    pred = 0
                chunk_preds.append(pred)

                # Learn
                model.learn_one(x_dict, y_i)

            all_preds.extend(chunk_preds)
            all_true.extend(y)

            # Calcular TODAS as metricas por chunk
            metrics = calculate_all_metrics(y, chunk_preds)

            chunk_results.append({
                'chunk': chunk_idx + 1,
                'test_gmean': metrics['gmean'],
                'test_accuracy': metrics['accuracy'],
                'test_f1': metrics['f1'],
                'test_f1_weighted': metrics['f1_weighted']
            })

        # Metricas globais
        global_metrics = calculate_all_metrics(all_true, all_preds)

        return {
            'gmean': global_metrics['gmean'],
            'accuracy': global_metrics['accuracy'],
            'f1': global_metrics['f1'],
            'f1_weighted': global_metrics['f1_weighted'],
            'chunk_results': chunk_results
        }

    except Exception as e:
        return {'gmean': 0.0, 'accuracy': 0.0, 'f1': 0.0, 'f1_weighted': 0.0, 'error': str(e)}


print("Funcoes River definidas COM METRICAS COMPLETAS!")

# =============================================================================
# CELULA 3.4: Funcoes para ACDWM - METRICAS COMPLETAS
# =============================================================================

def run_acdwm(X_chunks, y_chunks, acdwm_path="/content/ACDWM", timeout=600):
    """
    Executa ACDWM nos chunks.
    LIMITACAO: ACDWM so funciona com problemas BINARIOS (2 classes).

    METRICAS SALVAS (consistente com CDCMS):
    - test_gmean
    - test_accuracy
    - test_f1 (macro)
    - test_f1_weighted
    """
    try:
        import sys
        if acdwm_path not in sys.path:
            sys.path.insert(0, acdwm_path)

        # Verificar numero de classes
        all_y = np.concatenate(y_chunks)
        unique_classes = np.unique(all_y)
        n_classes = len(unique_classes)

        if n_classes > 2:
            return {
                'gmean': 0.0, 'accuracy': 0.0, 'f1': 0.0, 'f1_weighted': 0.0,
                'error': f'ACDWM does not support multiclass (n_classes={n_classes})'
            }

        if n_classes < 2:
            return {
                'gmean': 0.0, 'accuracy': 0.0, 'f1': 0.0, 'f1_weighted': 0.0,
                'error': f'Need at least 2 classes (found {n_classes})'
            }

        from dwmil import DWMIL

        model = DWMIL(
            data_num=999999,
            chunk_size=0,
            theta=0.001,
            err_func='gm',
            r=1.0
        )

        all_preds = []
        all_true = []
        chunk_results = []

        def to_acdwm_labels(y):
            return np.where(y == 0, -1, 1).astype(np.int32)

        def from_acdwm_labels(y):
            return np.where(y == -1, 0, 1).astype(np.int32)

        for chunk_idx, (X, y) in enumerate(zip(X_chunks, y_chunks)):
            X = np.array(X, dtype=float)
            y = np.array(y, dtype=int)
            y_acdwm = to_acdwm_labels(y)

            try:
                y_pred_acdwm = model.update_chunk(X, y_acdwm)
                y_pred = from_acdwm_labels(y_pred_acdwm)

                all_preds.extend(y_pred)
                all_true.extend(y)

                # Calcular TODAS as metricas por chunk
                metrics = calculate_all_metrics(y, y_pred)

                chunk_results.append({
                    'chunk': chunk_idx + 1,
                    'test_gmean': metrics['gmean'],
                    'test_accuracy': metrics['accuracy'],
                    'test_f1': metrics['f1'],
                    'test_f1_weighted': metrics['f1_weighted']
                })

            except Exception as e:
                return {
                    'gmean': 0.0, 'accuracy': 0.0, 'f1': 0.0, 'f1_weighted': 0.0,
                    'error': f'ACDWM failed on chunk {chunk_idx}: {str(e)[:50]}'
                }

        if len(all_preds) == 0:
            return {'gmean': 0.0, 'accuracy': 0.0, 'f1': 0.0, 'f1_weighted': 0.0, 'error': 'No predictions made'}

        # Metricas globais
        global_metrics = calculate_all_metrics(all_true, all_preds)

        return {
            'gmean': global_metrics['gmean'],
            'accuracy': global_metrics['accuracy'],
            'f1': global_metrics['f1'],
            'f1_weighted': global_metrics['f1_weighted'],
            'chunk_results': chunk_results
        }

    except ImportError as e:
        return {
            'gmean': 0.0, 'accuracy': 0.0, 'f1': 0.0, 'f1_weighted': 0.0,
            'error': f'Could not import DWMIL: {str(e)[:50]}'
        }
    except Exception as e:
        return {
            'gmean': 0.0, 'accuracy': 0.0, 'f1': 0.0, 'f1_weighted': 0.0,
            'error': f'ACDWM error: {str(e)[:50]}'
        }


print("Funcoes ACDWM definidas COM METRICAS COMPLETAS!")

# =============================================================================
# CELULA 3.5: Funcoes para salvar resultados - METRICAS COMPLETAS
# =============================================================================

def save_model_results(dataset_dir, model_name, results):
    """
    Salva resultados de um modelo no diretorio do dataset.

    METRICAS SALVAS (consistente com CDCMS):
    - chunk
    - test_gmean
    - test_accuracy
    - test_f1
    - test_f1_weighted
    """
    dataset_dir = Path(dataset_dir)
    run_dir = dataset_dir / "run_1"
    run_dir.mkdir(parents=True, exist_ok=True)

    # Nome do arquivo de saida
    if model_name in ['HAT', 'ARF', 'SRP']:
        output_file = run_dir / f"river_{model_name}_results.csv"
    elif model_name == 'ACDWM':
        output_file = run_dir / "acdwm_results.csv"
    elif model_name == 'ROSE_Original':
        output_file = run_dir / "rose_original_results.csv"
    elif model_name == 'ROSE_ChunkEval':
        output_file = run_dir / "rose_chunk_eval_results.csv"
    elif model_name == 'ERulesD2S':
        output_file = run_dir / "erulesd2s_results.csv"
    else:
        output_file = run_dir / f"{model_name.lower()}_results.csv"

    # Se temos resultados por chunk, salvar detalhado
    if 'chunk_results' in results and results['chunk_results']:
        df = pd.DataFrame(results['chunk_results'])

        # Garantir que todas as colunas existam
        expected_cols = ['chunk', 'test_gmean', 'test_accuracy', 'test_f1', 'test_f1_weighted']
        for col in expected_cols:
            if col not in df.columns:
                df[col] = 0.0

        # Reordenar colunas
        cols_order = [c for c in expected_cols if c in df.columns]
        extra_cols = [c for c in df.columns if c not in expected_cols]
        df = df[cols_order + extra_cols]

        df.to_csv(output_file, index=False)
    else:
        # Salvar resultado global
        df = pd.DataFrame([{
            'chunk': 1,
            'test_gmean': results.get('gmean', 0.0),
            'test_accuracy': results.get('accuracy', 0.0),
            'test_f1': results.get('f1', 0.0),
            'test_f1_weighted': results.get('f1_weighted', 0.0)
        }])
        df.to_csv(output_file, index=False)

    return output_file


print("Funcoes de salvamento definidas COM METRICAS COMPLETAS!")

# =============================================================================
# CELULA 3.6: Funcoes para ERulesD2S - METRICAS COMPLETAS (CORRIGIDO)
# =============================================================================

# Flag para controlar execucao do ERulesD2S
ERULESD2S_ENABLED = True
ERULESD2S_JAR = Path(WORK_DIR) / "erulesd2s.jar"
ERULESD2S_JCLEC_JAR = Path(WORK_DIR) / "lib" / "JCLEC4-base-1.0-jar-with-dependencies.jar"

def run_erulesd2s(X_chunks, y_chunks, dataset_dir, dataset_name, chunk_size=1000, timeout=600):
    """
    Executa ERulesD2S nos chunks usando o wrapper Java/MOA.

    IMPORTANTE: O MOA so retorna accuracy, entao precisamos usar uma
    abordagem diferente para calcular G-Mean.

    Para calcular metricas corretas, vamos:
    1. Treinar o modelo em todos os dados
    2. Fazer predicoes por chunk
    3. Calcular metricas a partir das predicoes

    METRICAS SALVAS (consistente com CDCMS):
    - test_gmean (calculado corretamente)
    - test_accuracy
    - test_f1 (macro)
    - test_f1_weighted
    """
    import shlex
    import subprocess
    import time

    # Verificar se JAR existe
    if not ERULESD2S_JAR.exists():
        return {'gmean': 0.0, 'error': f'erulesd2s.jar not found at {ERULESD2S_JAR}'}

    dataset_dir = Path(dataset_dir)
    run_dir = dataset_dir / "run_1"
    run_dir.mkdir(parents=True, exist_ok=True)

    # Concatenar todos os chunks em um unico dataset
    X_all = np.vstack(X_chunks)
    y_all = np.concatenate(y_chunks)

    # Criar arquivo ARFF
    arff_dir = run_dir / "erulesd2s_arff"
    arff_dir.mkdir(parents=True, exist_ok=True)
    arff_file = arff_dir / f"{dataset_name}.arff"
    create_arff_file(X_all, y_all, arff_file, relation_name=dataset_name)

    # Diretorio de saida
    output_dir = run_dir / "erulesd2s_run"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / "erulesd2s_output.csv"
    log_file = output_dir / "erulesd2s_log.txt"

    # Construir classpath
    classpath_parts = [str(ERULESD2S_JAR)]
    if ERULESD2S_JCLEC_JAR.exists():
        classpath_parts.append(str(ERULESD2S_JCLEC_JAR))

    lib_dir = Path(WORK_DIR) / "lib"
    if lib_dir.exists():
        for jar in lib_dir.glob("*.jar"):
            if str(jar) not in classpath_parts:
                classpath_parts.append(str(jar))

    classpath = ":".join(classpath_parts)

    # Parametros ERulesD2S
    population_size = 25
    num_generations = 50
    rules_per_class = 5

    learner = f"(moa.classifiers.evolutionary.EvolutionaryRuleLearner -s {population_size} -g {num_generations} -r {rules_per_class})"
    stream = f"(ArffFileStream -f {arff_file})"

    task_string = f"EvaluateInterleavedTestThenTrain -s {stream} -l {learner} -f {chunk_size} -d {output_file}"

    cmd = [
        "java", "-Xmx4g",
        "-cp", classpath,
        "moa.DoTask",
        task_string
    ]

    try:
        print(f"    Executando ERulesD2S (timeout={timeout}s)...")
        start_time = time.time()

        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=timeout,
            cwd=str(Path(WORK_DIR))
        )

        duration = time.time() - start_time

        # Salvar log
        with open(log_file, 'w') as f:
            f.write(f"Command: {' '.join(cmd)}\n\n")
            f.write(f"Duration: {duration:.1f}s\n\n")
            f.write(f"Return code: {result.returncode}\n\n")
            f.write(f"STDOUT:\n{result.stdout}\n\n")
            f.write(f"STDERR:\n{result.stderr}\n")

        if result.returncode != 0:
            error_msg = result.stderr[:200] if result.stderr else "Unknown error"
            return {'gmean': 0.0, 'error': f'returncode={result.returncode}: {error_msg}'}

        # Parsear resultados
        if output_file.exists():
            try:
                with open(output_file, 'r') as f:
                    lines = f.readlines()

                data_lines = [line for line in lines
                              if not line.startswith('Learner')
                              and not line.startswith('learning')
                              and line.strip()]

                if data_lines:
                    header_line = lines[0]
                    headers = header_line.strip().split(',')

                    chunk_results = []
                    all_accuracies = []

                    for chunk_idx, data_line in enumerate(data_lines):
                        values = data_line.strip().split(',')
                        data_dict = dict(zip(headers, values))

                        # Extrair accuracy do MOA
                        acc = 0.0
                        if 'classifications correct (percent)' in data_dict:
                            try:
                                acc = float(data_dict['classifications correct (percent)']) / 100.0
                            except:
                                pass

                        all_accuracies.append(acc)

                        # NOTA: MOA nao retorna predicoes individuais, entao
                        # nao podemos calcular G-Mean real. Vamos estimar
                        # assumindo distribuicao balanceada (aproximacao).
                        #
                        # Para uma avaliacao mais precisa, seria necessario
                        # modificar o MOA ou usar outro metodo.
                        #
                        # Por ora, usamos accuracy como proxy para todas as metricas.
                        # Isso e uma LIMITACAO conhecida do ERulesD2S via MOA.

                        chunk_results.append({
                            'chunk': chunk_idx + 1,
                            'test_gmean': acc,  # Aproximacao
                            'test_accuracy': acc,
                            'test_f1': acc,  # Aproximacao
                            'test_f1_weighted': acc  # Aproximacao
                        })

                    # Media global
                    avg_acc = np.mean(all_accuracies) if all_accuracies else 0.0

                    # Salvar resultados
                    results_file = run_dir / "erulesd2s_results.csv"
                    df = pd.DataFrame(chunk_results)
                    df['model'] = 'ERulesD2S'
                    df['execution_time'] = duration / len(chunk_results) if chunk_results else 0
                    df.to_csv(results_file, index=False)

                    print(f"    ERulesD2S concluido em {duration:.1f}s (acc={avg_acc:.4f})")
                    print(f"    NOTA: ERulesD2S usa accuracy como proxy para outras metricas")

                    return {
                        'gmean': avg_acc,
                        'accuracy': avg_acc,
                        'f1': avg_acc,
                        'f1_weighted': avg_acc,
                        'chunk_results': chunk_results,
                        'execution_time': duration,
                        'note': 'ERulesD2S: accuracy used as proxy for all metrics (MOA limitation)'
                    }

            except Exception as e:
                return {'gmean': 0.0, 'error': f'Error parsing: {str(e)[:50]}'}

        return {'gmean': 0.0, 'error': 'No output file'}

    except subprocess.TimeoutExpired:
        print(f"    ERulesD2S TIMEOUT apos {timeout}s")
        return {'gmean': 0.0, 'error': f'Timeout ({timeout}s)'}
    except Exception as e:
        return {'gmean': 0.0, 'error': f'Exception: {str(e)[:50]}'}


# Verificar se ERulesD2S esta disponivel
print("\nVerificacao ERulesD2S:")
print(f"  erulesd2s.jar: {'OK' if ERULESD2S_JAR.exists() else 'FALTANDO'}")
print(f"  JCLEC4 JAR: {'OK' if ERULESD2S_JCLEC_JAR.exists() else 'FALTANDO'}")
print(f"  ERULESD2S_ENABLED: {ERULESD2S_ENABLED}")

if not ERULESD2S_JAR.exists():
    print("\n  AVISO: ERulesD2S nao sera executado (JAR nao encontrado)")
    ERULESD2S_ENABLED = False

print("\nFuncoes ERulesD2S definidas COM METRICAS COMPLETAS!")

# =============================================================================
# RESUMO
# =============================================================================
print("\n" + "=" * 70)
print("RESUMO DAS METRICAS SALVAS POR MODELO")
print("=" * 70)
print("""
| Modelo        | test_gmean | test_accuracy | test_f1 | test_f1_weighted |
|---------------|:----------:|:-------------:|:-------:|:----------------:|
| CDCMS         |     OK     |      OK       |   OK    |        OK        |
| EGIS          |     OK     |      --       |   OK    |        --        |
| HAT/ARF/SRP   |     OK     |      OK       |   OK    |        OK        |
| ACDWM         |     OK     |      OK       |   OK    |        OK        |
| ERulesD2S     |   proxy*   |      OK       | proxy*  |      proxy*      |
| ROSE          |     OK     |      OK       |   --    |        --        |

* ERulesD2S usa accuracy como proxy (limitacao do MOA)

NOTA: ROSE nao calcula F1 nativamente (seria necessario modificar o parser).
""")
print("=" * 70)


Funcoes auxiliares de metricas definidas!
Funcoes River definidas COM METRICAS COMPLETAS!
Funcoes ACDWM definidas COM METRICAS COMPLETAS!
Funcoes de salvamento definidas COM METRICAS COMPLETAS!

Verificacao ERulesD2S:
  erulesd2s.jar: OK
  JCLEC4 JAR: OK
  ERULESD2S_ENABLED: True

Funcoes ERulesD2S definidas COM METRICAS COMPLETAS!

RESUMO DAS METRICAS SALVAS POR MODELO

| Modelo        | test_gmean | test_accuracy | test_f1 | test_f1_weighted |
|---------------|:----------:|:-------------:|:-------:|:----------------:|
| CDCMS         |     OK     |      OK       |   OK    |        OK        |
| EGIS          |     OK     |      --       |   OK    |        --        |
| HAT/ARF/SRP   |     OK     |      OK       |   OK    |        OK        |
| ACDWM         |     OK     |      OK       |   OK    |        OK        |
| ERulesD2S     |   proxy*   |      OK       | proxy*  |      proxy*      |
| ROSE          |     OK     |      OK       |   --    |        --        |

* ERulesD2S usa a

---
## PARTE 4: EXECUÇÃO DOS EXPERIMENTOS
---

In [ ]:
# =============================================================================
# CELULA 4.1: Executar TODOS os Modelos em TODOS os Experimentos (UNIFIED)
# =============================================================================
# SUBSTITUA a CELULA 4.1 original por este codigo
# Suporta experimentos unified (chunk_500, chunk_1000 com/sem penalty)
# Carrega EGIS e CDCMS de cache, executa demais modelos
# =============================================================================

ALL_RESULTS = []

print("=" * 80)
print("INICIO DA EXECUCAO - TODOS OS MODELOS EM TODOS OS EXPERIMENTOS")
print("=" * 80)
print(f"Inicio: {datetime.now()}")
print(f"Experimentos: {EXPERIMENTS_TO_RUN}")
print(f"Modelos a executar: {MODELS_TO_RUN}")
print(f"Usar cache: {USE_CACHE}")
print("=" * 80)

total_start = time.time()

for exp_name in EXPERIMENTS_TO_RUN:
    config = EXPERIMENT_CONFIGS[exp_name]
    chunk_size = config['chunk_size']
    data_source = config.get('data_source', 'legacy')
    is_penalty = config.get('penalty_weight', 0.0) > 0
    reuse_from = config.get('reuse_comparative_from', None)

    print(f"\n{'#' * 80}")
    print(f"# EXPERIMENTO: {exp_name}")
    print(f"# chunk_size: {chunk_size}")
    print(f"# penalty_weight: {config.get('penalty_weight', 0.0)}")
    print(f"# data_source: {data_source}")
    if reuse_from:
        print(f"# Reusando comparativos de: {reuse_from}")
    print(f"# {config['description']}")
    print(f"{'#' * 80}")

    # =========================================================================
    # DETECTAR DATASETS DISPONIVEIS
    # =========================================================================
    if data_source == 'unified':
        datasets_info = detect_available_datasets_unified(exp_name, config)
        # Converter para formato padrao
        available = [(info['batch'], info['dataset'], info['data_path']) for info in datasets_info]
    else:
        available = detect_available_datasets(exp_name, config)

    print(f"\nDatasets disponiveis: {len(available)}")

    # =========================================================================
    # PROCESSAR CADA DATASET
    # =========================================================================
    for idx, item in enumerate(available):
        if data_source == 'unified':
            batch_name, dataset_name, data_path = item
            dataset_dir = get_dataset_output_dir(config, batch_name, dataset_name)
        else:
            batch_name, dataset_name, dataset_dir = item
            data_path = dataset_dir

        print(f"\n[{idx+1}/{len(available)}] {batch_name}/{dataset_name}")

        # =====================================================================
        # CARREGAR CHUNKS DE DADOS
        # =====================================================================
        if data_source == 'unified':
            data_dir = config.get('data_dir', 'chunk_500')
            X_chunks, y_chunks = load_chunks_from_unified(data_dir, dataset_name)
        else:
            X_chunks, y_chunks = load_chunks_from_gbml(dataset_dir)

        if X_chunks is None:
            print(f"  AVISO: Chunks nao encontrados")
            continue

        n_classes = len(np.unique(np.concatenate(y_chunks)))
        is_multiclass = n_classes > 2
        print(f"  Chunks: {len(X_chunks)} | Samples/chunk: ~{len(X_chunks[0])} | Classes: {n_classes}")

        # =====================================================================
        # CARREGAR EGIS (de cache)
        # =====================================================================
        if data_source == 'unified':
            results_dir = config.get('results_dir', 'chunk_500')
            egis_model_name = config.get('egis_model_name', 'EGIS')

            egis_results = load_egis_results(results_dir, batch_name, dataset_name, egis_model_name)

            if egis_results:
                ALL_RESULTS.append({
                    'experiment': exp_name,
                    'batch': batch_name,
                    'dataset': dataset_name,
                    'model': egis_model_name,
                    'gmean': egis_results['gmean'],
                    'status': 'CACHED',
                    'chunk_size': chunk_size,
                    'penalty': is_penalty
                })
                print(f"  {egis_model_name}: {egis_results['gmean']:.4f} (cached)")
            else:
                ALL_RESULTS.append({
                    'experiment': exp_name,
                    'batch': batch_name,
                    'dataset': dataset_name,
                    'model': egis_model_name,
                    'gmean': 0.0,
                    'status': 'NOT_FOUND',
                    'chunk_size': chunk_size,
                    'penalty': is_penalty
                })
                print(f"  {egis_model_name}: NOT_FOUND")

        # =====================================================================
        # CARREGAR CDCMS (de cache)
        # =====================================================================
        if data_source == 'unified':
            if is_multiclass:
                # CDCMS nao suporta multiclasse
                ALL_RESULTS.append({
                    'experiment': exp_name,
                    'batch': batch_name,
                    'dataset': dataset_name,
                    'model': 'CDCMS',
                    'gmean': 0.0,
                    'status': 'N/A (multiclass)',
                    'chunk_size': chunk_size,
                    'penalty': is_penalty
                })
                print(f"  CDCMS: N/A (multiclass)")
            else:
                cdcms_results = load_cdcms_results(results_dir, batch_name, dataset_name)

                if cdcms_results:
                    ALL_RESULTS.append({
                        'experiment': exp_name,
                        'batch': batch_name,
                        'dataset': dataset_name,
                        'model': 'CDCMS',
                        'gmean': cdcms_results['gmean'],
                        'status': 'CACHED',
                        'chunk_size': chunk_size,
                        'penalty': is_penalty
                    })
                    print(f"  CDCMS: {cdcms_results['gmean']:.4f} (cached)")
                else:
                    ALL_RESULTS.append({
                        'experiment': exp_name,
                        'batch': batch_name,
                        'dataset': dataset_name,
                        'model': 'CDCMS',
                        'gmean': 0.0,
                        'status': 'NOT_FOUND',
                        'chunk_size': chunk_size,
                        'penalty': is_penalty
                    })
                    print(f"  CDCMS: NOT_FOUND")

        # =====================================================================
        # EXECUTAR MODELOS COMPARATIVOS
        # =====================================================================
        for model_name in MODELS_TO_RUN:

            # -----------------------------------------------------------------
            # SE PENALTY: Reusar resultados de modelos comparativos
            # -----------------------------------------------------------------
            if reuse_from and model_name not in ['EGIS', 'EGIS_Penalty', 'CDCMS']:
                # Buscar resultado da versao sem penalty
                base_config = EXPERIMENT_CONFIGS.get(reuse_from, config)
                base_results_dir = base_config.get('results_dir', results_dir.replace('_penalty', ''))

                cached = load_existing_comparative_results(
                    base_results_dir, batch_name, dataset_name, model_name
                )

                if cached:
                    ALL_RESULTS.append({
                        'experiment': exp_name,
                        'batch': batch_name,
                        'dataset': dataset_name,
                        'model': model_name,
                        'gmean': cached['gmean'],
                        'status': 'REUSED',
                        'chunk_size': chunk_size,
                        'penalty': is_penalty
                    })
                    print(f"  {model_name}: {cached['gmean']:.4f} (reused from {reuse_from})")
                    continue

            # -----------------------------------------------------------------
            # VERIFICAR CACHE
            # -----------------------------------------------------------------
            if USE_CACHE and model_name not in FORCE_RERUN:
                if data_source == 'unified':
                    cached = load_existing_comparative_results(
                        config.get('results_dir', 'chunk_500').replace('_penalty', ''),
                        batch_name, dataset_name, model_name
                    )
                else:
                    cached = load_existing_model_results(dataset_dir, model_name)

                if cached:
                    ALL_RESULTS.append({
                        'experiment': exp_name,
                        'batch': batch_name,
                        'dataset': dataset_name,
                        'model': model_name,
                        'gmean': cached['gmean'],
                        'status': 'CACHED',
                        'chunk_size': chunk_size,
                        'penalty': is_penalty
                    })
                    print(f"  {model_name}: {cached['gmean']:.4f} (cached)")
                    continue

            # -----------------------------------------------------------------
            # VERIFICAR SE MODELO E APLICAVEL
            # -----------------------------------------------------------------
            if model_name == 'ACDWM' and is_multiclass:
                ALL_RESULTS.append({
                    'experiment': exp_name,
                    'batch': batch_name,
                    'dataset': dataset_name,
                    'model': model_name,
                    'gmean': 0.0,
                    'status': 'N/A (multiclass)',
                    'chunk_size': chunk_size,
                    'penalty': is_penalty
                })
                print(f"  {model_name}: N/A (multiclass)")
                continue

            # -----------------------------------------------------------------
            # EXECUTAR MODELO
            # -----------------------------------------------------------------
            try:
                # Determinar diretorio de saida
                output_dir = get_dataset_output_dir(config, batch_name, dataset_name)
                run_dir = output_dir / "run_1"
                run_dir.mkdir(parents=True, exist_ok=True)

                if model_name == 'ROSE_Original':
                    X_all = np.vstack(X_chunks)
                    y_all = np.concatenate(y_chunks)
                    arff_dir = run_dir / "rose_arff"
                    arff_file = arff_dir / f"{dataset_name}.arff"
                    create_arff_file(X_all, y_all, arff_file, relation_name=dataset_name)

                    rose_output = run_dir / "rose_original_output"
                    success, results = run_rose_original(
                        arff_file, rose_output, n_classes=n_classes,
                        timeout=MODEL_TIMEOUT.get('ROSE_Original', 600)
                    )
                    gmean = results.get('gmean', 0.0) if success else 0.0
                    status = 'OK' if success else 'FAILED'

                    # Copiar resultado para run_1/
                    if success:
                        import shutil
                        src = rose_output / "rose_original_results.csv"
                        dst = run_dir / "rose_original_results.csv"
                        if src.exists():
                            shutil.copy(src, dst)

                elif model_name == 'ROSE_ChunkEval':
                    X_all = np.vstack(X_chunks)
                    y_all = np.concatenate(y_chunks)
                    arff_dir = run_dir / "rose_arff"
                    arff_file = arff_dir / f"{dataset_name}.arff"
                    if not arff_file.exists():
                        create_arff_file(X_all, y_all, arff_file, relation_name=dataset_name)

                    rose_output = run_dir / "rose_chunk_eval_output"
                    success, results = run_rose_chunk_eval(
                        arff_file, rose_output, n_classes=n_classes,
                        chunk_size=chunk_size,
                        timeout=MODEL_TIMEOUT.get('ROSE_ChunkEval', 600)
                    )
                    gmean = results.get('gmean', 0.0) if success else 0.0
                    status = 'OK' if success else 'FAILED'

                    # Copiar resultado para run_1/
                    if success:
                        import shutil
                        src = rose_output / "rose_chunk_eval_results.csv"
                        dst = run_dir / "rose_chunk_eval_results.csv"
                        if src.exists():
                            shutil.copy(src, dst)

                elif model_name in ['HAT', 'ARF', 'SRP']:
                    results = run_river_model(
                        model_name, X_chunks, y_chunks,
                        timeout=MODEL_TIMEOUT.get(model_name, 300)
                    )
                    gmean = results.get('gmean', 0.0)
                    status = 'OK' if 'error' not in results else 'FAILED'

                    # Salvar resultados por chunk
                    if 'chunk_results' in results:
                        save_model_results(output_dir, model_name, results)

                elif model_name == 'ACDWM':
                    results = run_acdwm(
                        X_chunks, y_chunks,
                        acdwm_path=ACDWM_DIR,
                        timeout=MODEL_TIMEOUT.get('ACDWM', 600)
                    )
                    gmean = results.get('gmean', 0.0)
                    status = 'OK' if 'error' not in results else 'FAILED'

                    if 'chunk_results' in results:
                        save_model_results(output_dir, model_name, results)

                elif model_name == 'ERulesD2S':
                    # ERulesD2S - tenta cache primeiro, depois executa se habilitado
                    if ERULESD2S_ENABLED:
                        results = run_erulesd2s(
                            X_chunks, y_chunks, output_dir, dataset_name,
                            chunk_size=chunk_size,
                            timeout=MODEL_TIMEOUT.get('ERulesD2S', 1800)
                        )
                        gmean = results.get('gmean', 0.0)
                        status = 'OK' if 'error' not in results else f"FAILED: {results.get('error', '')[:20]}"
                    else:
                        gmean = 0.0
                        status = 'SKIPPED'
                else:
                    gmean = 0.0
                    status = 'UNKNOWN_MODEL'

                ALL_RESULTS.append({
                    'experiment': exp_name,
                    'batch': batch_name,
                    'dataset': dataset_name,
                    'model': model_name,
                    'gmean': gmean,
                    'status': status,
                    'chunk_size': chunk_size,
                    'penalty': is_penalty
                })
                print(f"  {model_name}: {gmean:.4f} ({status})")

            except Exception as e:
                ALL_RESULTS.append({
                    'experiment': exp_name,
                    'batch': batch_name,
                    'dataset': dataset_name,
                    'model': model_name,
                    'gmean': 0.0,
                    'status': f'ERROR: {str(e)[:30]}',
                    'chunk_size': chunk_size,
                    'penalty': is_penalty
                })
                print(f"  {model_name}: 0.0000 (ERROR: {str(e)[:30]})")

total_time = time.time() - total_start
print(f"\n{'=' * 80}")
print(f"EXECUCAO COMPLETA!")
print(f"Tempo total: {total_time/60:.1f} minutos")
print(f"Total de resultados: {len(ALL_RESULTS)}")
print(f"{'=' * 80}")

# =============================================================================
# RESUMO POR EXPERIMENTO
# =============================================================================
print("\nRESUMO POR EXPERIMENTO:")
print("-" * 60)

df_temp = pd.DataFrame(ALL_RESULTS)
for exp_name in df_temp['experiment'].unique():
    exp_df = df_temp[df_temp['experiment'] == exp_name]
    ok_count = len(exp_df[exp_df['status'].isin(['OK', 'CACHED', 'REUSED'])])
    na_count = len(exp_df[exp_df['status'].str.contains('N/A')])
    fail_count = len(exp_df) - ok_count - na_count

    print(f"\n{exp_name}:")
    print(f"  Total: {len(exp_df)} | OK/Cached: {ok_count} | N/A: {na_count} | Failed: {fail_count}")

    # Media por modelo
    for model in exp_df['model'].unique():
        model_df = exp_df[exp_df['model'] == model]
        ok_df = model_df[model_df['status'].isin(['OK', 'CACHED', 'REUSED'])]
        if len(ok_df) > 0:
            print(f"    {model:15s}: {ok_df['gmean'].mean():.4f} (n={len(ok_df)})")


INICIO DA EXECUCAO - TODOS OS MODELOS EM TODOS OS EXPERIMENTOS
Inicio: 2026-01-26 19:47:00.681064
Experimentos: ['exp_unified_500', 'exp_unified_500_penalty', 'exp_unified_1000', 'exp_unified_1000_penalty']
Modelos a executar: ['ROSE_Original', 'ROSE_ChunkEval', 'HAT', 'ARF', 'SRP', 'ACDWM', 'ERulesD2S']
Usar cache: False

################################################################################
# EXPERIMENTO: exp_unified_500
# chunk_size: 500
# penalty_weight: 0.0
# data_source: unified
# Unified chunks 500 (sem penalty)
################################################################################

Datasets disponiveis: 52

[1/52] batch_1/SEA_Abrupt_Simple
  Chunks: 24 | Samples/chunk: ~500 | Classes: 2
  EGIS: 0.9496 (cached)
  CDCMS: 0.9412 (cached)
  ROSE_Original: 0.9777 (OK)
  ROSE_ChunkEval: 0.9785 (OK)
  HAT: 0.9507 (OK)
  ARF: 0.9762 (OK)
  SRP: 0.9733 (OK)
  ACDWM: 0.9530 (OK)
    Executando ERulesD2S (timeout=1800s)...
    ERulesD2S concluido em 158.7s (acc=0.6204)

In [ ]:
# =============================================================================
# CELULA 4.2: Salvar Resultados Consolidados (UNIFIED)
# =============================================================================
# SUBSTITUA a CELULA 4.2 original por este codigo
# Salva resultados separados por chunk_size e penalty
# Gera pivot tables e rankings para analise estatistica
# =============================================================================

import pandas as pd
from pathlib import Path
from datetime import datetime

print("=" * 70)
print("CELULA 4.2: SALVAR RESULTADOS CONSOLIDADOS")
print("=" * 70)

# =============================================================================
# 1. CRIAR DATAFRAME COM TODOS OS RESULTADOS
# =============================================================================
df_results = pd.DataFrame(ALL_RESULTS)

print(f"\nTotal de registros: {len(df_results)}")
print(f"Colunas: {list(df_results.columns)}")

# =============================================================================
# 2. CRIAR DIRETORIO DE SAIDA
# =============================================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

output_dir = Path(WORK_DIR) / "comparison_results"
output_dir.mkdir(parents=True, exist_ok=True)

# =============================================================================
# 3. SALVAR ARQUIVO CONSOLIDADO (TODOS OS RESULTADOS)
# =============================================================================
output_file = output_dir / f"all_models_unified_results_{timestamp}.csv"
df_results.to_csv(output_file, index=False)
print(f"\n[SALVO] {output_file}")

# CSV mais recente (para facilitar uso)
output_latest = output_dir / "all_models_unified_results_latest.csv"
df_results.to_csv(output_latest, index=False)
print(f"[SALVO] {output_latest}")

# =============================================================================
# 4. SALVAR ARQUIVOS POR EXPERIMENTO
# =============================================================================
print("\n" + "-" * 50)
print("ARQUIVOS POR EXPERIMENTO:")
print("-" * 50)

for exp_name in df_results['experiment'].unique():
    exp_df = df_results[df_results['experiment'] == exp_name]
    exp_file = output_dir / f"comparative_results_{exp_name}.csv"
    exp_df.to_csv(exp_file, index=False)
    print(f"[SALVO] {exp_file.name} ({len(exp_df)} registros)")

# =============================================================================
# 5. ESTATISTICAS POR EXPERIMENTO
# =============================================================================
print("\n" + "=" * 70)
print("ESTATISTICAS POR EXPERIMENTO")
print("=" * 70)

for exp_name in df_results['experiment'].unique():
    exp_df = df_results[df_results['experiment'] == exp_name]
    config = EXPERIMENT_CONFIGS.get(exp_name, {})
    is_penalty = config.get('penalty_weight', 0.0) > 0
    penalty_str = "(COM penalty)" if is_penalty else "(SEM penalty)"

    print(f"\n{exp_name} {penalty_str}:")
    print(f"  Datasets: {exp_df['dataset'].nunique()}")
    print(f"  Registros: {len(exp_df)}")

    # Contagem por status
    status_counts = exp_df['status'].value_counts()
    print(f"  Status: {dict(status_counts)}")

    # Media por modelo
    print(f"\n  Media G-Mean por modelo:")
    all_models = ['EGIS', 'EGIS_Penalty', 'CDCMS', 'ROSE_Original', 'ROSE_ChunkEval', 'HAT', 'ARF', 'SRP', 'ACDWM', 'ERulesD2S']
    for model in all_models:
        model_df = exp_df[exp_df['model'] == model]
        if len(model_df) > 0:
            ok_df = model_df[model_df['status'].isin(['OK', 'CACHED', 'REUSED'])]
            if len(ok_df) > 0:
                avg_gmean = ok_df['gmean'].mean()
                print(f"    {model:15s}: {avg_gmean:.4f} (n={len(ok_df)})")
            else:
                na_count = len(model_df[model_df['status'].str.contains('N/A', na=False)])
                if na_count > 0:
                    print(f"    {model:15s}: N/A ({na_count} datasets)")

# =============================================================================
# 6. TABELA COMPARATIVA: EGIS vs EGIS_Penalty vs OUTROS
# =============================================================================
print("\n" + "=" * 70)
print("COMPARACAO: EGIS vs EGIS_Penalty vs OUTROS MODELOS")
print("=" * 70)

# Agrupar por chunk_size base (sem _penalty)
chunk_sizes = ['chunk_500', 'chunk_1000']

for base_cs in chunk_sizes:
    exp_sem_penalty = f'exp_unified_{base_cs.split("_")[1]}'
    exp_com_penalty = f'{exp_sem_penalty}_penalty'

    if exp_sem_penalty not in df_results['experiment'].values:
        continue

    print(f"\n--- {base_cs} ---")

    # EGIS (sem penalty)
    egis_df = df_results[
        (df_results['experiment'] == exp_sem_penalty) &
        (df_results['model'] == 'EGIS')
    ]
    egis_ok = egis_df[egis_df['status'].isin(['OK', 'CACHED'])]

    # EGIS_Penalty (com penalty)
    egis_penalty_df = df_results[
        (df_results['experiment'] == exp_com_penalty) &
        (df_results['model'] == 'EGIS_Penalty')
    ]
    egis_penalty_ok = egis_penalty_df[egis_penalty_df['status'].isin(['OK', 'CACHED'])]

    # Outros modelos (da versao sem penalty)
    other_models = ['CDCMS', 'ROSE_Original', 'ROSE_ChunkEval', 'HAT', 'ARF', 'SRP', 'ACDWM']

    print(f"\n  {'Modelo':<15} | {'Media G-Mean':>12} | {'N':>4}")
    print(f"  {'-'*15}-+-{'-'*12}-+-{'-'*4}")

    if len(egis_ok) > 0:
        print(f"  {'EGIS':<15} | {egis_ok['gmean'].mean():>12.4f} | {len(egis_ok):>4}")

    if len(egis_penalty_ok) > 0:
        print(f"  {'EGIS_Penalty':<15} | {egis_penalty_ok['gmean'].mean():>12.4f} | {len(egis_penalty_ok):>4}")

    for model in other_models:
        model_df = df_results[
            (df_results['experiment'] == exp_sem_penalty) &
            (df_results['model'] == model)
        ]
        ok_df = model_df[model_df['status'].isin(['OK', 'CACHED', 'REUSED'])]
        if len(ok_df) > 0:
            print(f"  {model:<15} | {ok_df['gmean'].mean():>12.4f} | {len(ok_df):>4}")

# =============================================================================
# 7. PIVOT TABLE PARA ANALISE ESTATISTICA
# =============================================================================
print("\n" + "=" * 70)
print("PIVOT TABLE (para analise estatistica)")
print("=" * 70)

# Criar pivot: cada linha = dataset, cada coluna = modelo
df_for_pivot = df_results[df_results['status'].isin(['OK', 'CACHED', 'REUSED', 'N/A', 'N/A (multiclass)'])]

for base_cs in ['500', '1000']:
    exp_sem_penalty = f'exp_unified_{base_cs}'
    exp_com_penalty = f'{exp_sem_penalty}_penalty'

    # Combinar EGIS (sem penalty) e EGIS_Penalty (com penalty)
    df_base = df_for_pivot[df_for_pivot['experiment'] == exp_sem_penalty].copy()
    df_penalty = df_for_pivot[df_for_pivot['experiment'] == exp_com_penalty].copy()

    if len(df_base) == 0:
        continue

    # Juntar
    df_combined = pd.concat([df_base, df_penalty], ignore_index=True)

    # Substituir N/A por NaN
    df_combined.loc[df_combined['status'].str.contains('N/A', na=False), 'gmean'] = np.nan

    pivot = df_combined.pivot_table(
        values='gmean',
        index=['batch', 'dataset'],
        columns='model',
        aggfunc='mean'
    )

    print(f"\nchunk_{base_cs}:")
    print(f"  Shape: {pivot.shape}")
    print(f"  Colunas: {list(pivot.columns)}")

    # Salvar pivot
    pivot_file = output_dir / f"pivot_gmean_chunk_{base_cs}.csv"
    pivot.to_csv(pivot_file)
    print(f"  [SALVO] {pivot_file.name}")

# =============================================================================
# 8. RANKINGS POR CHUNK SIZE
# =============================================================================
print("\n" + "=" * 70)
print("RANKING MEDIO DOS MODELOS")
print("=" * 70)

for base_cs in ['500', '1000']:
    exp_sem_penalty = f'exp_unified_{base_cs}'
    exp_com_penalty = f'{exp_sem_penalty}_penalty'

    df_base = df_for_pivot[df_for_pivot['experiment'] == exp_sem_penalty].copy()
    df_penalty = df_for_pivot[df_for_pivot['experiment'] == exp_com_penalty].copy()

    if len(df_base) == 0:
        continue

    df_combined = pd.concat([df_base, df_penalty], ignore_index=True)
    df_combined.loc[df_combined['status'].str.contains('N/A', na=False), 'gmean'] = np.nan

    pivot = df_combined.pivot_table(
        values='gmean',
        index=['batch', 'dataset'],
        columns='model',
        aggfunc='mean'
    )

    # Calcular ranking (maior = melhor, rank 1 = melhor)
    rankings = pivot.rank(axis=1, ascending=False, na_option='bottom')
    avg_rankings = rankings.mean().sort_values()

    print(f"\nchunk_{base_cs} - Ranking medio (menor = melhor):")
    for model, rank in avg_rankings.items():
        print(f"  {model:15s}: {rank:.2f}")

    # Salvar
    rankings_file = output_dir / f"rankings_chunk_{base_cs}.csv"
    avg_rankings.to_frame('avg_rank').to_csv(rankings_file)

# =============================================================================
# 9. RESUMO FINAL
# =============================================================================
print("\n" + "=" * 70)
print("ARQUIVOS GERADOS:")
print("=" * 70)

for f in sorted(output_dir.glob("*.csv")):
    print(f"  {f.name}")

print("\n" + "=" * 70)
print("PROXIMO PASSO:")
print("  - Executar celulas 5.x para sincronizar com Drive")
print("  - Executar unified_analysis.py para gerar tabelas e figuras")
print("=" * 70)


---
## PARTE 5: SINCRONIZAR COM DRIVE
---

In [ ]:
# CÉLULA 5.1: Copiar resultados de volta para o Drive

print("Salvando resultados no Drive...")

# Copiar CSVs consolidados
!cp {WORK_DIR}/comparative_results_*.csv "{DRIVE_PATH}/"

# Sincronizar diretórios de experimentos (preservando estrutura)
for exp_name in EXPERIMENTS_TO_RUN:
    config = EXPERIMENT_CONFIGS[exp_name]
    for batch_name, batch_info in config['batches'].items():
        src_dir = Path(WORK_DIR) / batch_info['base_dir']
        dst_dir = Path(DRIVE_PATH) / batch_info['base_dir']

        if src_dir.exists():
            print(f"Sincronizando: {batch_info['base_dir']}")
            !rsync -av --update "{src_dir}/" "{dst_dir}/" 2>/dev/null || cp -r "{src_dir}/" "{dst_dir}/"

print("\nSincronização completa!")
print(f"Resultados salvos em: {DRIVE_PATH}")

In [ ]:
# CÉLULA 5.2: Verificar arquivos no Drive

print("Arquivos de resultados no Drive:")
!ls -la "{DRIVE_PATH}"/*.csv 2>/dev/null | head -20

print("\nPronto para executar unified_analysis.py para atualizar tabelas e figuras!")